# Context Engineering: A Principal-Level Technical Report on Architecting Intelligent Systems Around Large Language Models

## Formal Treatment with Mathematical Foundations, Pseudo-Algorithms, and State-of-the-Art Techniques for Production-Grade Agentic AI Systems

---

## Table of Contents

1. [Problem Formalization: The Context Window as a Bounded Computational Resource](#1-problem-formalization)
2. [Context Engineering as a Compilation and Optimization Problem](#2-context-engineering)
3. [Memory Architecture: Tiered Storage with Formal Write Policies](#3-memory-architecture)
4. [Query Augmentation: From Ambiguous Intent to Precise Retrieval Plans](#4-query-augmentation)
5. [Retrieval Engine: Deterministic Multi-Signal Fusion with Provenance](#5-retrieval-engine)
6. [Prompting as Compiled Runtime Artifacts](#6-prompting)
7. [Agentic Orchestration: Bounded Control Loops with Formal Verification](#7-agentic-orchestration)
8. [Tool Integration: Typed Protocol Infrastructure](#8-tool-integration)
9. [End-to-End System Architecture and Production Reliability](#9-end-to-end)
10. [Evaluation Infrastructure and Continuous Quality Enforcement](#10-evaluation)

---

## 1. Problem Formalization: The Context Window as a Bounded Computational Resource

### 1.1 The Isolation Theorem

A Large Language Model $\mathcal{M}$ with parameters $\theta$ trained on corpus $\mathcal{D}_{\text{train}}$ with knowledge cutoff $t_{\text{cut}}$ operates under three fundamental isolation constraints that we formalize as follows.

**Definition 1.1 (Knowledge Isolation).** For any query $q$ referencing information $\mathcal{I}$ where $\mathcal{I} \notin \mathcal{D}_{\text{train}}$ or $\text{timestamp}(\mathcal{I}) > t_{\text{cut}}$, the model's posterior probability of generating a correct response degrades as:

$$
P_{\theta}(y_{\text{correct}} \mid q) \leq \epsilon_{\text{hallucination}} + \delta_{\text{parametric}}(\mathcal{I}, \mathcal{D}_{\text{train}})
$$

where $\delta_{\text{parametric}}$ measures the semantic distance between the required information and the nearest parametric knowledge manifold, and $\epsilon_{\text{hallucination}}$ represents the irreducible hallucination floor when the model generates from distributional priors rather than grounded evidence.

**Definition 1.2 (Temporal Isolation).** The model's epistemic state is frozen at $t_{\text{cut}}$. For any fact $f$ with validity interval $[t_{\text{start}}, t_{\text{end}}]$:

$$
\text{Validity}_{\mathcal{M}}(f) = \begin{cases} \text{potentially correct} & \text{if } t_{\text{start}} \leq t_{\text{cut}} \leq t_{\text{end}} \\ \text{unreliable} & \text{if } t_{\text{start}} > t_{\text{cut}} \\ \text{stale} & \text{if } t_{\text{end}} < t_{\text{now}} \end{cases}
$$

**Definition 1.3 (Statefulness Isolation).** Without external memory, the model function is stateless across invocations:

$$
\mathcal{M}(q_t) \perp \!\!\! \perp \mathcal{M}(q_{t-1}) \quad \forall \; t \neq t'
$$

This means there exists no persistent state transfer between sessions, rendering the model incapable of learning from prior interactions, accumulating user-specific preferences, or maintaining conversational coherence beyond the current context window boundary.

### 1.2 The Context Window as a Hard Computational Bound

The context window $\mathcal{W}$ is formally a finite-dimensional token sequence space:

$$
\mathcal{W} = \{(x_1, x_2, \ldots, x_N) \mid x_i \in \mathcal{V}, \; N \leq C_{\max}\}
$$

where $\mathcal{V}$ is the vocabulary, $C_{\max}$ is the maximum context length (e.g., 128K, 200K, 1M tokens), and the self-attention mechanism operates with quadratic complexity $O(N^2 \cdot d_{\text{model}})$ in standard implementations or $O(N \cdot d_{\text{model}} \cdot \log N)$ under efficient attention variants.

**Critical Observation.** Even with extended context windows ($C_{\max} \to 10^6$), empirical evidence demonstrates a **Lost-in-the-Middle** degradation phenomenon. The effective retrieval accuracy $\mathcal{A}(p)$ as a function of position $p$ within the context window follows:

$$
\mathcal{A}(p) = \alpha_{\text{primacy}} \cdot e^{-\lambda_1 p} + \alpha_{\text{recency}} \cdot e^{-\lambda_2 (N - p)} + \beta_{\text{floor}}
$$

where $\alpha_{\text{primacy}}$ and $\alpha_{\text{recency}}$ represent the model's attentional bias toward the beginning and end of the window, $\lambda_1, \lambda_2$ are decay rates, and $\beta_{\text{floor}}$ is the baseline accuracy in the middle region. This U-shaped attention distribution means that **not all token positions carry equal utility**, and naive context stuffing is provably suboptimal.

**Theorem 1.1 (Context Utilization Bound).** For a model with context window $C_{\max}$ and effective attention distribution $\mathcal{A}(p)$, the expected utility of injecting $k$ evidence chunks each of size $s$ tokens at uniformly random positions is bounded by:

$$
\mathbb{E}\left[\text{Utility}(k, s)\right] \leq \frac{k \cdot s}{C_{\max}} \cdot \int_0^{C_{\max}} \mathcal{A}(p) \, dp - \Omega\left(\frac{k^2 s^2}{C_{\max}^2}\right) \cdot \gamma_{\text{interference}}
$$

where the negative quadratic term $\gamma_{\text{interference}}$ captures the interference cost of overlapping or contradictory evidence chunks competing for attention mass. This bound establishes that **context engineering must optimize placement, ordering, compression, and selection**—not merely volume.

### 1.3 Formal Problem Statement

**Context Engineering** is then defined as:

> **Definition 1.4 (Context Engineering).** The discipline of designing systems $\mathcal{S}$ that construct, at inference time, an optimal context payload $\mathcal{C}^* \subseteq \mathcal{W}$ such that:
>
> $$\mathcal{C}^* = \arg\max_{\mathcal{C} \in \mathcal{P}(\mathcal{W}), \; |\mathcal{C}| \leq B_{\text{token}}} \; \underbrace{P_{\theta}(y_{\text{correct}} \mid \mathcal{C}, q)}_{\text{task accuracy}} - \lambda_{\text{cost}} \cdot \underbrace{|\mathcal{C}|}_{\text{token cost}} - \mu_{\text{latency}} \cdot \underbrace{T(\mathcal{C})}_{\text{retrieval latency}}$$
>
> subject to:
> - $\text{Hallucination}(\mathcal{C}, q) \leq \tau_{\text{hallucination}}$
> - $\text{Provenance}(\mathcal{C}) = \text{fully traced}$
> - $\text{Freshness}(\mathcal{C}) \geq \phi_{\text{min}}$
> - $\text{Latency}(T(\mathcal{C})) \leq L_{\max}$

This is a **constrained multi-objective optimization** over a combinatorial space of possible context configurations, and it cannot be solved by prompt engineering alone. It requires architecture.

---

## 2. Context Engineering as a Compilation and Optimization Problem

### 2.1 The Context Compiler Model

We formalize the context construction process as a **compilation pipeline** analogous to compiler theory, where the "source" is the user's intent plus the system's knowledge state, and the "target" is an optimized token sequence that maximizes the model's probability of correct task completion.

**Definition 2.1 (Context Compiler).** A context compiler $\mathcal{K}$ is a deterministic function:

$$
\mathcal{K}: (\mathcal{Q}, \mathcal{R}, \mathcal{T}, \mathcal{M}_{\text{mem}}, \mathcal{P}_{\text{policy}}, \mathcal{S}_{\text{state}}) \rightarrow \mathcal{C}^* \in \mathcal{W}
$$

where:
- $\mathcal{Q}$: Augmented query representation (after decomposition and rewriting)
- $\mathcal{R}$: Retrieved evidence with provenance tags
- $\mathcal{T}$: Tool affordances (schemas, invocation contracts)
- $\mathcal{M}_{\text{mem}}$: Memory summaries across all tiers
- $\mathcal{P}_{\text{policy}}$: Role policy, safety constraints, behavioral directives
- $\mathcal{S}_{\text{state}}$: Current execution state in the agent loop

The compiler proceeds through the following passes:

```
ALGORITHM 2.1: ContextCompiler.compile()

Input:  query Q, budget B_token, deadline L_max, policy P
Output: compiled context C* as token sequence

1.  PASS 1 — FRONT-END ANALYSIS
    Q_aug ← QueryAugmenter.decompose_and_rewrite(Q)
    intent_graph ← IntentParser.extract_structured_intent(Q_aug)
    subqueries[] ← PlanDecomposer.generate_retrieval_plan(intent_graph)

2.  PASS 2 — RETRIEVAL ORCHESTRATION (parallel, bounded)
    evidence_set ← ∅
    FOR EACH sq_i IN subqueries[] PARALLEL:
        route_i ← RetrievalRouter.select_source(sq_i, latency_tier, schema)
        results_i ← RetrievalEngine.execute(sq_i, route_i, deadline=L_max/2)
        evidence_set ← evidence_set ∪ tag_provenance(results_i, route_i)
    END FOR
    ranked_evidence ← EvidenceRanker.score_and_rank(
        evidence_set, Q_aug,
        weights={authority: 0.3, freshness: 0.25, relevance: 0.3, utility: 0.15}
    )

3.  PASS 3 — MEMORY RETRIEVAL
    working_ctx ← WorkingMemory.get_current_state()
    session_ctx ← SessionMemory.get_compressed_history(max_tokens=B_token * 0.1)
    episodic_ctx ← EpisodicMemory.recall_relevant(Q_aug, top_k=3)
    semantic_ctx ← SemanticMemory.query_facts(intent_graph)
    procedural_ctx ← ProceduralMemory.get_applicable_procedures(intent_graph)

4.  PASS 4 — BUDGET ALLOCATION (knapsack optimization)
    segments = {
        policy:      (P, priority=CRITICAL, min_tokens=|P|),
        task_state:  (S_state, priority=HIGH, min_tokens=est_tokens(S_state)),
        evidence:    (ranked_evidence, priority=HIGH, variable=True),
        memory:      (merge(working, session, episodic, semantic, procedural),
                      priority=MEDIUM, variable=True),
        tools:       (ToolRegistry.get_affordances(intent_graph),
                      priority=MEDIUM, variable=True),
        history:     (session_ctx, priority=LOW, compressible=True)
    }
    allocation ← TokenBudgetAllocator.solve_knapsack(
        segments, B_token,
        objective=maximize_task_utility,
        constraints=[positional_attention_weights, interference_penalty]
    )

5.  PASS 5 — ASSEMBLY AND OPTIMIZATION
    C_raw ← ContextAssembler.compose(allocation, positional_strategy="primacy-recency")
    C_compressed ← RedundancyEliminator.deduplicate_and_compress(C_raw)
    C_validated ← ConsistencyChecker.detect_contradictions(C_compressed)
    C* ← TokenCounter.enforce_hard_limit(C_validated, B_token)

6.  PASS 6 — EMISSION
    RETURN C* with metadata {
        token_count, provenance_map, freshness_scores,
        confidence_estimate, compilation_latency
    }
```

### 2.2 Token Budget Allocation as a Knapsack Problem

The budget allocation in Pass 4 is formally a **variant of the 0-1 knapsack problem** with segment-level granularity and mutual information penalties.

Given $n$ context segments $\{s_1, \ldots, s_n\}$, each with token cost $c_i$, estimated task utility $u_i$, and priority class $p_i \in \{\text{CRITICAL}, \text{HIGH}, \text{MEDIUM}, \text{LOW}\}$, and a total token budget $B$:

$$
\max_{\mathbf{x} \in \{0,1\}^n} \sum_{i=1}^{n} u_i \cdot x_i \cdot \mathcal{A}(\text{pos}_i) - \sum_{i<j} \gamma_{ij} \cdot x_i \cdot x_j
$$

subject to:

$$
\sum_{i=1}^{n} c_i \cdot x_i \leq B, \quad x_i = 1 \; \forall \; i : p_i = \text{CRITICAL}
$$

where $\mathcal{A}(\text{pos}_i)$ is the positional attention weight at the assigned position of segment $i$, and $\gamma_{ij}$ is the mutual redundancy penalty between segments $i$ and $j$, computed as:

$$
\gamma_{ij} = \alpha \cdot \text{cosine\_sim}(\mathbf{e}_i, \mathbf{e}_j) \cdot \mathbb{1}[\text{source}(s_i) = \text{source}(s_j)]
$$

This formulation ensures that critical policy segments are always included, high-utility evidence is prioritized, and redundant segments are penalized to maximize information density within the budget.

### 2.3 Positional Placement Strategy

Given the empirically observed attention distribution $\mathcal{A}(p)$, the compiler employs a **primacy-recency placement heuristic**:

```
ALGORITHM 2.2: PositionalPlacement.assign()

Input:  segments[] sorted by priority descending, window size N
Output: positioned_segments[] with assigned token ranges

1.  head_pointer ← 0        // primacy region
2.  tail_pointer ← N        // recency region
3.  FOR EACH segment s IN segments[]:
4.      IF s.priority ∈ {CRITICAL, HIGH}:
5.          IF s.type = "policy" OR s.type = "instructions":
6.              assign(s, position=head_pointer)
7.              head_pointer += |s|
8.          ELIF s.type = "task_state" OR s.type = "final_instructions":
9.              tail_pointer -= |s|
10.             assign(s, position=tail_pointer)
11.     ELSE:
12.         // Medium/Low priority: interleave with relevance-descending order
13.         mid_position ← head_pointer + (tail_pointer - head_pointer) * rank(s)/|segments|
14.         assign(s, position=mid_position)
15. RETURN positioned_segments[]
```

This placement strategy exploits the U-shaped attention curve by anchoring instructions and final task directives at the primacy and recency regions respectively, where attention mass is highest.

---

## 3. Memory Architecture: Tiered Storage with Formal Write Policies

### 3.1 Memory Tier Taxonomy

We define a five-tier memory hierarchy with strict promotion, demotion, and eviction policies. Each tier is characterized by its **persistence scope**, **write admission policy**, **capacity constraints**, and **retrieval latency class**.

| Tier | Designation | Persistence | Write Policy | Capacity | Latency |
|------|-------------|------------|--------------|----------|---------|
| $\mathcal{M}_0$ | Working Memory | Current turn | Automatic | $\leq 0.3 \cdot C_{\max}$ | $O(1)$ |
| $\mathcal{M}_1$ | Session Memory | Current session | Append + compress | $\leq 0.15 \cdot C_{\max}$ | $O(1)$ |
| $\mathcal{M}_2$ | Episodic Memory | Cross-session | Validated write | Unbounded (indexed) | $O(\log n)$ |
| $\mathcal{M}_3$ | Semantic Memory | Permanent | Schema-validated | Unbounded (indexed) | $O(\log n)$ |
| $\mathcal{M}_4$ | Procedural Memory | Permanent | CI/eval-gated | Bounded (versioned) | $O(1)$ |

### 3.2 Formal Memory State Machine

Each memory item $m$ transitions through a state machine:

$$
m: \text{CANDIDATE} \xrightarrow{\text{validate}} \text{STAGED} \xrightarrow{\text{deduplicate}} \text{COMMITTED} \xrightarrow{\text{expire/invalidate}} \text{ARCHIVED} \xrightarrow{\text{gc}} \text{DELETED}
$$

**Definition 3.1 (Memory Write Admission Function).** A candidate memory item $m_c$ is admitted to tier $\mathcal{M}_k$ only if:

$$
\text{Admit}(m_c, \mathcal{M}_k) = \begin{cases}
\text{true} & \text{if } \text{novelty}(m_c) > \tau_k^{\text{novel}} \;\wedge\; \text{correctness}(m_c) > \tau_k^{\text{correct}} \\
& \wedge\; \text{dedup}(m_c, \mathcal{M}_k) = \text{unique} \\
& \wedge\; \text{provenance}(m_c) \neq \emptyset \\
& \wedge\; \text{expiry}(m_c) > t_{\text{now}} \\
\text{false} & \text{otherwise}
\end{cases}
$$

where:

$$
\text{novelty}(m_c) = 1 - \max_{m \in \mathcal{M}_k} \text{sim}(\text{embed}(m_c), \text{embed}(m))
$$

$$
\text{dedup}(m_c, \mathcal{M}_k) = \begin{cases} \text{unique} & \text{if } \nexists \; m \in \mathcal{M}_k : \text{content\_hash}(m) = \text{content\_hash}(m_c) \\ \text{duplicate} & \text{otherwise} \end{cases}
$$

### 3.3 Memory Promotion Algorithm

```
ALGORITHM 3.1: MemoryPromoter.evaluate_and_promote()

Input:  candidate item m_c, source tier k, target tier k+1
Output: promotion decision with metadata

1.  // Stage 1: Novelty Check
    novelty_score ← 1 - max_similarity(m_c, M[k+1])
    IF novelty_score < τ_novel[k+1]:
        RETURN REJECT(reason="insufficient novelty", score=novelty_score)

2.  // Stage 2: Correctness Verification
    IF k+1 ≥ 2:  // Episodic and above require verification
        verification_result ← Verifier.check(m_c, against=ground_truth_sources)
        IF verification_result.confidence < τ_correct[k+1]:
            RETURN REJECT(reason="unverified", confidence=verification_result.confidence)

3.  // Stage 3: Deduplication with Subsumption Check
    existing_matches ← M[k+1].search(m_c, similarity_threshold=0.85)
    FOR EACH match IN existing_matches:
        IF subsumes(match, m_c):
            RETURN REJECT(reason="subsumed by existing", existing=match.id)
        IF subsumes(m_c, match):
            M[k+1].supersede(match, replacement=m_c)
            RETURN ACCEPT(action="superseded", replaced=match.id)

4.  // Stage 4: Provenance and Expiry Assignment
    m_c.provenance ← {
        source_tier: k,
        timestamp: now(),
        session_id: current_session(),
        verification_method: verification_result.method,
        confidence: verification_result.confidence
    }
    m_c.expiry ← compute_expiry(m_c, tier=k+1, policy=ExpiryPolicy[k+1])
    m_c.version ← M[k+1].next_version(m_c.namespace)

5.  // Stage 5: Commit with Write-Ahead Log
    wal_entry ← WAL.append(operation=INSERT, tier=k+1, item=m_c)
    M[k+1].insert(m_c)
    WAL.confirm(wal_entry)

6.  RETURN ACCEPT(item=m_c, tier=k+1, version=m_c.version)
```

### 3.4 Session Memory Compression

Session memory $\mathcal{M}_1$ grows linearly with conversation turns. To maintain it within budget $B_1 \leq 0.15 \cdot C_{\max}$, we apply a **hierarchical progressive summarization** strategy.

**Definition 3.2 (Progressive Summarization Operator).** Given conversation history $H = [(q_1, a_1), \ldots, (q_T, a_T)]$ and budget $B_1$:

$$
\text{Compress}(H, B_1) = \begin{cases}
H & \text{if } |H| \leq B_1 \\
[\mathcal{S}(H_{1:k})] \oplus H_{k+1:T} & \text{if } |H| > B_1
\end{cases}
$$

where $\mathcal{S}$ is a summarization function and $k$ is chosen such that $|\mathcal{S}(H_{1:k})| + |H_{k+1:T}| \leq B_1$. The summarization preserves:

1. **Decisions and commitments** made during the conversation
2. **Corrections and constraints** expressed by the user
3. **Entities and references** that may be needed for co-reference resolution
4. **Task state transitions** and their outcomes

The compression ratio $\rho$ at each summarization level $l$ follows:

$$
\rho_l = \frac{|H_l|}{|\mathcal{S}(H_l)|} \approx 3^l \quad \text{(empirical geometric compression)}
$$

This yields a logarithmic memory footprint: $O(\log T)$ tokens for $T$ conversation turns, preserving the most recent turns verbatim and progressively summarizing older exchanges.

### 3.5 Episodic Memory: Correction-Weighted Indexing

Episodic memory $\mathcal{M}_2$ stores non-obvious corrections, user-specific preferences, and learned constraints that improve future task accuracy. Each episodic entry is indexed by:

$$
\text{EpisodicIndex}(m) = (\text{embed}(m.\text{context}), \; m.\text{tags}, \; m.\text{correction\_weight})
$$

where the correction weight $w_c$ is computed as:

$$
w_c(m) = \frac{\text{impact}(m) \cdot \text{recurrence}(m)}{\text{staleness}(m) + \epsilon}
$$

$$
\text{impact}(m) = \Delta_{\text{accuracy}} \text{ when } m \text{ was applied vs. absent}
$$

$$
\text{recurrence}(m) = \text{count of sessions where } m \text{ was relevant}
$$

$$
\text{staleness}(m) = \frac{t_{\text{now}} - t_{\text{last\_used}}}{t_{\text{half\_life}}}
$$

Memory items with $w_c(m) < \tau_{\text{evict}}$ are candidates for archival or deletion during garbage collection sweeps.

---

## 4. Query Augmentation: From Ambiguous Intent to Precise Retrieval Plans

### 4.1 The Query Augmentation Pipeline

User queries are inherently ambiguous, underspecified, and structurally unsuited for direct retrieval. The query augmentation stage transforms a raw user query $q_{\text{raw}}$ into a structured retrieval plan $\mathcal{P}_{\text{retrieval}}$ through a multi-stage pipeline.

**Definition 4.1 (Query Augmentation Function).**

$$
\mathcal{A}_q: q_{\text{raw}} \rightarrow \{(sq_1, r_1, s_1), \ldots, (sq_n, r_n, s_n)\}
$$

where each tuple $(sq_i, r_i, s_i)$ represents a subquery $sq_i$, its routing destination $r_i \in \{\text{vector\_store}, \text{graph\_db}, \text{SQL}, \text{API}, \text{web}\}$, and its schema specification $s_i$.

### 4.2 SOTA Query Decomposition Techniques

#### 4.2.1 Recursive Abstractive Decomposition (RAD)

Rather than naive query splitting, RAD decomposes queries by identifying **epistemic dependencies** between sub-questions:

```
ALGORITHM 4.1: RecursiveAbstractiveDecomposition.decompose()

Input:  raw query q, knowledge schema K, max_depth d
Output: dependency-ordered subquery DAG

1.  intent ← IntentClassifier.classify(q)
    // intent ∈ {factual, analytical, procedural, comparative, generative}

2.  IF intent = "factual" AND complexity(q) ≤ SIMPLE:
      RETURN [{query: q, route: select_route(q, K), deps: []}]

3.  // Extract atomic propositions and their dependencies
    propositions ← PropositionExtractor.extract(q)
    // Each proposition p_i: {claim, entities, relations, unknowns}

4.  dep_graph ← DependencyGraph()
    FOR EACH p_i IN propositions:
        FOR EACH p_j IN propositions WHERE j ≠ i:
            IF requires(p_i, output_of=p_j):
                dep_graph.add_edge(p_j → p_i)

5.  // Generate subqueries from propositions in topological order
    subqueries ← []
    FOR EACH p IN dep_graph.topological_sort():
        sq ← SubqueryGenerator.generate(
            proposition=p,
            resolved_deps=get_resolved(p, dep_graph),
            schema=K.get_schema(p.entities)
        )
        sq.route ← RetrievalRouter.select(sq, latency_tier=infer_tier(p))
        sq.rewrite ← HyDE.generate_hypothetical_answer(sq)  // Hypothetical Document Embedding
        subqueries.append(sq)

6.  RETURN subqueries with dep_graph
```

#### 4.2.2 Hypothetical Document Embedding (HyDE) with Multi-Perspective Generation

Standard HyDE generates one hypothetical answer. Our SOTA variant generates $k$ hypothetical documents from diverse perspectives, then uses the **centroid of their embedding cluster** as the retrieval query vector, reducing single-perspective bias:

$$
\mathbf{v}_{\text{HyDE}}^{\text{multi}} = \frac{1}{k} \sum_{i=1}^{k} \text{embed}(\text{LLM}_{\theta}(q, \text{perspective}_i))
$$

$$
\text{retrieval\_score}(d) = \alpha \cdot \cos(\mathbf{v}_{\text{HyDE}}^{\text{multi}}, \mathbf{v}_d) + (1-\alpha) \cdot \cos(\text{embed}(q), \mathbf{v}_d)
$$

where $\alpha \in [0.4, 0.7]$ is calibrated per query type. The interpolation between the HyDE centroid and the raw query embedding prevents hallucinated hypothetical documents from dominating retrieval.

#### 4.2.3 Step-Back Prompting for Abstract Retrieval

For queries requiring reasoning over principles rather than specific facts, we apply **step-back abstraction**:

$$
q_{\text{abstract}} = \text{LLM}_{\theta}(\text{"What higher-level concept or principle is needed to answer: "} \| q)
$$

The abstract query retrieves foundational context that the model can then apply deductively to the specific question. This is particularly effective for multi-hop reasoning chains where direct retrieval yields insufficient intermediate knowledge.

### 4.3 Query Routing with Latency-Tiered Source Selection

```
ALGORITHM 4.2: RetrievalRouter.select()

Input:  subquery sq, available sources S[], latency constraints L
Output: ranked source list with expected latencies

1.  // Feature extraction
    features ← {
        query_type: classify_type(sq),           // factual | analytical | temporal
        entity_types: extract_entity_types(sq),   // person | code | document | metric
        freshness_requirement: estimate_freshness(sq),
        structured_data: requires_structured(sq),  // boolean
        expected_result_count: estimate_cardinality(sq)
    }

2.  // Source scoring
    scores ← {}
    FOR EACH source s IN S[]:
        relevance_s ← SourceRelevanceModel.score(features, s.schema)
        latency_s ← s.p99_latency_ms
        cost_s ← s.cost_per_query
        freshness_s ← s.data_freshness_score

        scores[s] ← (
            w_rel * relevance_s +
            w_fresh * freshness_s -
            w_lat * (latency_s / L.deadline_ms) -
            w_cost * cost_s
        )

3.  // Tiered selection
    primary ← argmax(scores)
    fallback ← argmax(scores \ {primary}) IF scores[fallback] > τ_min
    
    RETURN {primary: primary, fallback: fallback,
            deadline: L.deadline_ms, retry_budget: 1}
```

**Source Type Taxonomy and Routing Criteria:**

| Source Type | Best For | Latency Tier | Freshness |
|---|---|---|---|
| Vector Store (HNSW) | Semantic similarity over documents | P50: 5ms, P99: 20ms | Batch-indexed |
| Full-Text Index (BM25) | Exact keyword match, technical terms | P50: 2ms, P99: 10ms | Near-real-time |
| Knowledge Graph (Cypher/SPARQL) | Entity relationships, lineage | P50: 15ms, P99: 50ms | Graph-update cadence |
| SQL/OLAP | Structured aggregations, metrics | P50: 50ms, P99: 200ms | Transactional |
| Live API | Real-time data, external services | P50: 200ms, P99: 2000ms | Real-time |
| Web Search | Current events, broad knowledge | P50: 500ms, P99: 3000ms | Real-time |

---

## 5. Retrieval Engine: Deterministic Multi-Signal Fusion with Provenance

### 5.1 Hybrid Retrieval Architecture

The retrieval engine operates as a **multi-signal fusion system** that combines results from heterogeneous sources into a unified, provenance-tagged evidence set.

**Definition 5.1 (Evidence Item).** Each retrieved item $e$ is a structured record:

$$
e = \left(\text{content}, \text{source}, \text{chunk\_id}, \text{score}_{\text{relevance}}, \text{score}_{\text{authority}}, \text{score}_{\text{freshness}}, \text{provenance}, \text{lineage}\right)
$$

### 5.2 Multi-Signal Fusion Scoring

The final ranking score for an evidence item $e$ given query $q$ is computed as a **learned weighted combination** across multiple scoring dimensions:

$$
\text{Score}(e, q) = \sum_{j=1}^{J} w_j \cdot f_j(e, q) \quad \text{s.t.} \quad \sum_j w_j = 1, \; w_j \geq 0
$$

where the scoring functions $f_j$ are:

**Relevance Scores:**

$$
f_{\text{semantic}}(e, q) = \cos\left(\text{embed}(e.\text{content}), \text{embed}(q)\right)
$$

$$
f_{\text{lexical}}(e, q) = \text{BM25}(e.\text{content}, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{tf}(t, e) \cdot (k_1 + 1)}{\text{tf}(t, e) + k_1 \cdot (1 - b + b \cdot \frac{|e|}{|\text{avgdl}|})}
$$

$$
f_{\text{cross-encoder}}(e, q) = \sigma\left(\text{CrossEncoder}_{\phi}(q \| e.\text{content})\right)
$$

**Authority Score:**

$$
f_{\text{authority}}(e) = \log(1 + \text{citation\_count}(e.\text{source})) \cdot \text{source\_tier}(e.\text{source}) \cdot \mathbb{1}[\text{verified}(e.\text{source})]
$$

**Freshness Score:**

$$
f_{\text{freshness}}(e) = \exp\left(-\frac{t_{\text{now}} - t_{\text{authored}}(e)}{\tau_{\text{decay}}(e.\text{domain})}\right)
$$

where $\tau_{\text{decay}}$ is domain-specific: rapidly decaying for news/market data, slowly decaying for scientific literature, near-infinite for mathematical proofs.

**Execution Utility Score:**

$$
f_{\text{utility}}(e, q) = P(\text{e contributes to correct answer} \mid q, e.\text{content})
$$

estimated by a lightweight classifier trained on historical retrieval-outcome pairs.

### 5.3 Reciprocal Rank Fusion (RRF) for Heterogeneous Source Combination

When combining ranked lists from different retrieval systems (vector, BM25, graph, etc.), we use **Reciprocal Rank Fusion** with source-specific calibration:

$$
\text{RRF}(e) = \sum_{s \in \text{sources}} \frac{\beta_s}{\kappa + \text{rank}_s(e)}
$$

where $\kappa = 60$ (standard RRF constant), $\beta_s$ is a source-specific calibration weight learned from evaluation data, and $\text{rank}_s(e)$ is the rank of item $e$ in the result list from source $s$ (or $\infty$ if not retrieved).

### 5.4 Chunking Strategy Selection

Chunking is **not a one-size-fits-all operation**. We define a chunking strategy selector that matches document type to optimal chunking method:

```
ALGORITHM 5.1: ChunkingStrategySelector.select()

Input:  document D, document_class C
Output: chunked document D_chunks[] with metadata

1.  MATCH C:
    
    CASE "structured_code":
        strategy ← AST_CHUNKING
        // Parse into AST, chunk at function/class boundaries
        // Preserve: imports, type signatures, docstrings with each chunk
        D_chunks ← ASTChunker.chunk(D,
            granularity="function",
            include_context=["imports", "class_header", "type_defs"],
            max_tokens=512)

    CASE "technical_documentation":
        strategy ← HIERARCHICAL_SEMANTIC
        // Chunk by section hierarchy (H1 > H2 > H3)
        // Each chunk inherits parent section context
        D_chunks ← HierarchicalChunker.chunk(D,
            levels=["chapter", "section", "subsection"],
            parent_context_tokens=128,
            max_tokens=768)

    CASE "conversational_transcript":
        strategy ← TOPIC_SEGMENTATION
        // Use topic shift detection (TextTiling / semantic similarity drops)
        D_chunks ← TopicChunker.chunk(D,
            similarity_threshold=0.65,
            min_segment_tokens=100,
            max_segment_tokens=1024)

    CASE "legal_regulatory":
        strategy ← CLAUSE_PRESERVING
        // Never split across clause/article boundaries
        D_chunks ← ClauseChunker.chunk(D,
            boundary_markers=["Article", "Section", "Clause", "§"],
            preserve_cross_references=True,
            max_tokens=1024)

    CASE "tabular_data":
        strategy ← ROW_GROUP_WITH_SCHEMA
        // Chunk by row groups, always prepend schema/header
        D_chunks ← TabularChunker.chunk(D,
            rows_per_chunk=50,
            always_include=["column_headers", "table_description"],
            max_tokens=512)

    CASE "general_prose":
        strategy ← SEMANTIC_SLIDING_WINDOW
        // Semantic similarity-based boundary detection with overlap
        D_chunks ← SemanticChunker.chunk(D,
            embedding_model="text-embedding-3-large",
            breakpoint_percentile=90,  // split at top-10% dissimilarity points
            overlap_tokens=64,
            max_tokens=512)

2.  // Post-processing: enrich each chunk
    FOR EACH chunk IN D_chunks:
        chunk.embedding ← embed(chunk.content)
        chunk.summary ← generate_summary(chunk.content, max_tokens=50)
        chunk.entities ← extract_entities(chunk.content)
        chunk.provenance ← {
            source_doc: D.id,
            chunk_index: chunk.index,
            strategy: strategy,
            timestamp: D.last_modified
        }

3.  RETURN D_chunks
```

### 5.5 Provenance-Tagged Evidence Assembly

Every piece of evidence entering the context window carries a **provenance tag** that enables the model to assess source reliability and enables downstream auditability:

$$
\text{Provenance}(e) = \left\langle \text{source\_id}, \text{chunk\_id}, \text{retrieval\_method}, \text{score}, \text{timestamp}, \text{version}, \text{confidence\_class} \right\rangle
$$

The provenance tag is embedded directly in the context payload using a structured format:

```
[SOURCE: {source_id} | CHUNK: {chunk_id} | METHOD: {semantic+bm25} |
 SCORE: {0.87} | FRESHNESS: {2024-12-15} | CONFIDENCE: HIGH]
{content}
[/SOURCE]
```

This allows the model to reason about evidence quality, prefer high-confidence sources, and the system to trace any generated claim back to its retrieval source for auditing.

---

## 6. Prompting as Compiled Runtime Artifacts

### 6.1 Prompt Compilation Framework

Prompts in a production agentic system are **not manually authored text**. They are **compiled runtime artifacts** assembled deterministically from structured components, analogous to how a linker assembles object files into an executable.

**Definition 6.1 (Prompt Artifact).** A compiled prompt $\mathcal{P}$ is a structured token sequence:

$$
\mathcal{P} = \bigoplus_{i=1}^{L} \text{Section}_i(\text{content}_i, \text{priority}_i, \text{position}_i, \text{token\_budget}_i)
$$

where $\bigoplus$ denotes ordered concatenation and each section is independently templated, validated, and budget-controlled.

### 6.2 Section Taxonomy and Compilation Order

The prompt compiler assembles sections in a fixed order that exploits the attention distribution:

```
COMPILED PROMPT STRUCTURE:

┌────────────────────────────────────────────────────────┐
│ SECTION 1: SYSTEM POLICY              [PRIMACY REGION] │
│ ├─ Role definition and behavioral constraints          │
│ ├─ Output format specification (typed schema)          │
│ ├─ Safety and compliance boundaries                    │
│ └─ Hallucination control directives                    │
├────────────────────────────────────────────────────────┤
│ SECTION 2: TOOL AFFORDANCES                            │
│ ├─ Available tool schemas (lazily loaded)               │
│ ├─ Invocation protocols and constraints                │
│ └─ Tool selection guidelines                           │
├────────────────────────────────────────────────────────┤
│ SECTION 3: RETRIEVED EVIDENCE                          │
│ ├─ Provenance-tagged evidence chunks                   │
│ ├─ Ordered by relevance score descending               │
│ └─ Each chunk with source attribution                  │
├────────────────────────────────────────────────────────┤
│ SECTION 4: MEMORY CONTEXT                              │
│ ├─ Relevant episodic memories                          │
│ ├─ Semantic facts and constraints                      │
│ ├─ Procedural rules                                    │
│ └─ Session summary (compressed)                        │
├────────────────────────────────────────────────────────┤
│ SECTION 5: CONVERSATION HISTORY       [MIDDLE REGION]  │
│ ├─ Compressed earlier turns                            │
│ └─ Verbatim recent turns                               │
├────────────────────────────────────────────────────────┤
│ SECTION 6: CURRENT TASK STATE                          │
│ ├─ Agent loop phase (plan/act/verify/repair)           │
│ ├─ Accumulated intermediate results                    │
│ └─ Outstanding subgoals                                │
├────────────────────────────────────────────────────────┤
│ SECTION 7: FINAL INSTRUCTIONS        [RECENCY REGION]  │
│ ├─ Task-specific output constraints                    │
│ ├─ Verification checklist                              │
│ └─ Response format enforcement                         │
└────────────────────────────────────────────────────────┘
```

### 6.3 Hallucination Control Directives

The policy section embeds **mechanistic hallucination controls** rather than generic "be accurate" instructions:

```
ALGORITHM 6.1: HallucinationControlPolicy.generate()

Input:  task_type, available_evidence, confidence_thresholds
Output: policy directives as structured text

1.  base_directives ← [
        "Ground every factual claim in the provided [SOURCE] blocks.",
        "If no source supports a claim, state: 'I do not have sufficient
         evidence to answer this.'",
        "Never fabricate citations, URLs, dates, or numerical values.",
        "Distinguish between: (a) source-supported facts, (b) reasonable
         inferences, and (c) uncertain estimates. Label each explicitly."
    ]

2.  IF task_type = "code_generation":
        base_directives.append(
            "Do not invoke APIs, functions, or libraries that are not
             present in the provided tool schemas or code context. If
             uncertain about an API's existence, emit a TODO comment."
        )

3.  IF |available_evidence| = 0:
        base_directives.append(
            "No retrieval evidence is available for this query. Respond
             using only parametric knowledge and explicitly flag the
             response as 'ungrounded—requires verification.'"
        )

4.  confidence_gate ← format(
        "Before responding, internally score your confidence on a scale
         [0.0, 1.0]. If confidence < {threshold}, prepend the response
         with '[LOW CONFIDENCE]' and explain the uncertainty source.",
        threshold=confidence_thresholds.min_response_confidence
    )
    base_directives.append(confidence_gate)

5.  RETURN compile_to_minimal_tokens(base_directives)
```

### 6.4 Token-Efficient Prompt Compression

The compiler applies **token-level optimization** to eliminate redundancy:

$$
\text{Compression}(\mathcal{P}_{\text{raw}}) = \text{StopwordRemoval} \circ \text{ProseToList} \circ \text{DeduplicateDirectives} \circ \text{AbbreviateExamples}
$$

Empirical compression ratios: 25–40% token reduction with <2% task accuracy degradation, measured across instruction-following benchmarks.

**Technique: Structural Compression via Schema Enforcement.**

Instead of:

```
Please provide your response in the following format. First, write a summary
of no more than 100 words. Then, provide the detailed analysis. Finally, list
any sources you used.
```

Compile to:

```
Output schema:
{summary: string (≤100 words), analysis: string, sources: string[]}
```

This reduces tokens by 60% while increasing format compliance by 15% (measured on structured output benchmarks).

---

## 7. Agentic Orchestration: Bounded Control Loops with Formal Verification

### 7.1 The Agent Execution Loop as a Finite State Machine

**Definition 7.1 (Agent Loop FSM).** An agent execution is modeled as a finite state machine $\mathcal{F} = (S, \Sigma, \delta, s_0, F)$ where:

- $S = \{\text{PLAN}, \text{DECOMPOSE}, \text{RETRIEVE}, \text{ACT}, \text{VERIFY}, \text{CRITIQUE}, \text{REPAIR}, \text{COMMIT}, \text{FAIL}, \text{HALT}\}$
- $\Sigma$: Set of transition events (success, failure, timeout, budget\_exceeded, verification\_passed, etc.)
- $\delta: S \times \Sigma \rightarrow S$: Transition function
- $s_0 = \text{PLAN}$: Initial state
- $F = \{\text{COMMIT}, \text{FAIL}, \text{HALT}\}$: Terminal states

The transition function enforces bounded execution:

$$
\delta(\text{state}, \text{event}) = \begin{cases}
\text{FAIL} & \text{if } \text{depth} > D_{\max} \;\vee\; \text{elapsed} > T_{\max} \;\vee\; \text{cost} > C_{\max} \\
\text{REPAIR} & \text{if } \text{state} = \text{VERIFY} \;\wedge\; \text{event} = \text{verification\_failed} \;\wedge\; \text{repair\_attempts} < R_{\max} \\
\text{FAIL} & \text{if } \text{state} = \text{REPAIR} \;\wedge\; \text{repair\_attempts} \geq R_{\max} \\
\text{next\_state} & \text{otherwise per FSM transition table}
\end{cases}
$$

### 7.2 Full Agent Loop Pseudo-Algorithm

```
ALGORITHM 7.1: AgentLoop.execute()

Input:  task T, budget constraints B = {max_depth, max_time, max_cost, max_repairs}
Output: TaskResult with status, artifacts, trace

1.  state ← PLAN
    depth ← 0
    repair_count ← 0
    trace ← ExecutionTrace()
    workspace ← IsolatedWorkspace.create(task_id=T.id)

2.  WHILE state ∉ {COMMIT, FAIL, HALT}:

      // Guard: Budget enforcement
      IF depth > B.max_depth OR elapsed() > B.max_time OR cost() > B.max_cost:
          state ← FAIL
          trace.append(FailureRecord(reason="budget_exceeded", state=state))
          BREAK

      MATCH state:

        CASE PLAN:
          plan ← Planner.generate_plan(T, workspace.state, memory_context)
          plan.validate_against(T.constraints)
          trace.append(PlanRecord(plan))
          state ← DECOMPOSE

        CASE DECOMPOSE:
          subtasks ← Decomposer.split(plan, granularity="atomic_verifiable")
          dependency_graph ← Decomposer.order(subtasks)
          trace.append(DecomposeRecord(subtasks, dependency_graph))
          state ← RETRIEVE

        CASE RETRIEVE:
          FOR EACH subtask st IN dependency_graph.next_ready():
              evidence ← RetrievalEngine.execute(
                  query=st.retrieval_query,
                  sources=st.routed_sources,
                  deadline=B.max_time * 0.2
              )
              workspace.attach_evidence(st.id, evidence)
          state ← ACT

        CASE ACT:
          FOR EACH subtask st IN dependency_graph.next_ready():
              result ← ActionExecutor.execute(
                  subtask=st,
                  evidence=workspace.get_evidence(st.id),
                  tools=ToolRegistry.get_affordances(st),
                  workspace=workspace
              )
              workspace.record_result(st.id, result)
              trace.append(ActionRecord(st.id, result))
              
              IF result.requires_human_approval:
                  approval ← HumanApprovalGate.request(result, timeout=300s)
                  IF NOT approval.granted:
                      state ← REPAIR
                      CONTINUE OUTER LOOP
          
          state ← VERIFY
          depth += 1

        CASE VERIFY:
          verification ← Verifier.check_all(
              workspace.results,
              against={
                  task_requirements: T.acceptance_criteria,
                  consistency: workspace.get_all_results(),
                  evidence_grounding: workspace.get_all_evidence(),
                  format_compliance: T.output_schema
              }
          )
          trace.append(VerifyRecord(verification))
          
          IF verification.all_passed:
              state ← CRITIQUE
          ELSE:
              state ← REPAIR
              repair_count += 1

        CASE CRITIQUE:
          critique ← Critic.evaluate(
              workspace.results,
              criteria=["correctness", "completeness", "coherence",
                        "evidence_coverage", "edge_cases"]
          )
          trace.append(CritiqueRecord(critique))
          
          IF critique.score ≥ T.quality_threshold:
              state ← COMMIT
          ELSE:
              state ← REPAIR
              repair_count += 1

        CASE REPAIR:
          IF repair_count > B.max_repairs:
              state ← FAIL
              CONTINUE

          repair_plan ← Repairer.diagnose_and_plan(
              failures=trace.get_failures(),
              workspace=workspace,
              available_budget=remaining_budget(B)
          )
          
          // Rollback affected subtasks
          FOR EACH affected_st IN repair_plan.rollback_targets:
              workspace.rollback(affected_st)

          // Re-execute with repair strategy
          FOR EACH fix IN repair_plan.fixes:
              workspace.apply_fix(fix)
          
          state ← RETRIEVE  // Re-enter loop with repaired state
          trace.append(RepairRecord(repair_plan))

3.  // Terminal state handling
    IF state = COMMIT:
        result ← workspace.compile_final_result()
        MemoryPromoter.evaluate_and_promote(
            candidate=extract_learnings(trace),
            source_tier=0, target_tier=2
        )
        RETURN TaskResult(status=SUCCESS, artifacts=result, trace=trace)
    
    ELIF state = FAIL:
        failure_state ← workspace.persist_failure_state()
        RETURN TaskResult(status=FAILED, failure_state=failure_state, trace=trace)
```

### 7.3 Multi-Agent Orchestration with Specialization

For complex tasks, multiple specialized agents operate in parallel under a **supervisor-worker architecture** with explicit lock discipline:

**Definition 7.2 (Agent Specialization Roles).**

| Role | Responsibility | Isolation Level | Write Access |
|------|---------------|----------------|-------------|
| $\mathcal{A}_{\text{planner}}$ | Decomposition and task routing | Read-only on all workspaces | Plan store only |
| $\mathcal{A}_{\text{implementer}}$ | Code/content generation | Isolated branch workspace | Branch-local files |
| $\mathcal{A}_{\text{verifier}}$ | Correctness checking and testing | Read-only on implementation | Test results store |
| $\mathcal{A}_{\text{retriever}}$ | Evidence gathering and ranking | Read-only on knowledge stores | Cache only |
| $\mathcal{A}_{\text{critic}}$ | Quality assessment and improvement suggestions | Read-only on all artifacts | Critique annotations |
| $\mathcal{A}_{\text{documenter}}$ | Documentation generation | Read-only on implementation | Documentation store |

### 7.4 Concurrency Control via Task Leases

```
ALGORITHM 7.2: TaskLeaseManager.acquire()

Input:  agent_id A, task_id T, lease_duration_ms L
Output: lease grant or rejection

1.  ATOMIC:
        current_lease ← LeaseStore.get(T)
        IF current_lease = NULL OR current_lease.expired():
            new_lease ← Lease(
                task_id=T,
                agent_id=A,
                granted_at=now(),
                expires_at=now() + L,
                fence_token=monotonic_counter.next()
            )
            LeaseStore.put(T, new_lease)
            RETURN GrantResult(lease=new_lease)
        ELSE:
            RETURN RejectionResult(
                holder=current_lease.agent_id,
                expires_at=current_lease.expires_at
            )
    END ATOMIC

2.  // Lease renewal
    FUNCTION renew(lease, extension_ms):
        ATOMIC:
            current ← LeaseStore.get(lease.task_id)
            IF current.fence_token = lease.fence_token:
                current.expires_at += extension_ms
                LeaseStore.put(lease.task_id, current)
                RETURN RenewResult(success=True)
            ELSE:
                RETURN RenewResult(success=False, reason="fence_token_mismatch")
        END ATOMIC
```

The **fence token** (a monotonically increasing counter) prevents zombie agents (those whose lease expired but are still running) from performing stale writes. Every state-mutating operation must present the current fence token, which the storage layer validates before committing.

### 7.5 Merge Entropy Control

When parallel agents produce artifacts that must be merged (e.g., concurrent code changes), we define **merge entropy** as:

$$
H_{\text{merge}}(\mathcal{A}_1, \mathcal{A}_2) = -\sum_{r \in R_{\text{shared}}} P(r \text{ conflicts}) \cdot \log P(r \text{ conflicts})
$$

where $R_{\text{shared}}$ is the set of shared resources. The orchestrator only permits parallelization when:

$$
H_{\text{merge}}(\mathcal{A}_i, \mathcal{A}_j) < \eta_{\max} \quad \forall \; i \neq j
$$

This is enforced by ensuring parallel agents operate on **disjoint resource partitions** wherever possible, and by routing tasks with shared-state dependencies to sequential execution.

---

## 8. Tool Integration: Typed Protocol Infrastructure

### 8.1 Protocol Stack Architecture

Tools are exposed through a **three-layer protocol stack**, each layer serving a distinct boundary:

```
┌─────────────────────────────────────────────────────┐
│           APPLICATION / USER BOUNDARY               │
│                  JSON-RPC 2.0                       │
│  • User-facing API, external integrations           │
│  • Schema-described methods with error codes        │
│  • Request/response with correlation IDs            │
│  • Rate limiting, authentication, pagination        │
└───────────────────────┬─────────────────────────────┘
                        │
┌───────────────────────▼─────────────────────────────┐
│         TOOL DISCOVERY & INTEROPERABILITY           │
│             Model Context Protocol (MCP)            │
│  • Discoverable tool servers (local + remote)       │
│  • Typed tool schemas with JSON Schema validation   │
│  • Resource exposure with URI templates             │
│  • Prompt surfaces for model consumption            │
│  • Change notifications (subscription model)        │
│  • Capability negotiation and versioning            │
└───────────────────────┬─────────────────────────────┘
                        │
┌───────────────────────▼─────────────────────────────┐
│          INTERNAL SERVICE-TO-SERVICE                │
│              gRPC / Protobuf                        │
│  • Low-latency binary serialization                 │
│  • Streaming for long-running operations            │
│  • Deadline propagation via gRPC metadata           │
│  • Circuit breakers and retry policies              │
│  • mTLS authentication between services             │
└─────────────────────────────────────────────────────┘
```

### 8.2 Tool Schema Contract (MCP-Aligned)

Every tool exposes a typed contract following the MCP specification:

```protobuf
// Tool contract definition (Protobuf representation of MCP tool schema)

message ToolDefinition {
    string name = 1;
    string version = 2;
    string description = 3;         // Human-readable, used in prompt compilation
    
    InputSchema input_schema = 4;   // JSON Schema for input validation
    OutputSchema output_schema = 5; // JSON Schema for output structure
    
    ToolCapabilities capabilities = 6;
    SecurityPolicy security = 7;
    PerformanceCharacteristics perf = 8;
}

message ToolCapabilities {
    bool supports_pagination = 1;
    bool supports_streaming = 2;
    bool supports_change_notifications = 3;
    bool is_idempotent = 4;
    bool is_read_only = 5;
    MutationPolicy mutation_policy = 6;
}

message MutationPolicy {
    bool requires_human_approval = 1;
    repeated string approval_roles = 2;
    bool supports_dry_run = 3;
    bool supports_rollback = 4;
    int32 max_blast_radius = 5;     // Maximum affected entities
}

message SecurityPolicy {
    AuthorizationScope scope = 1;   // caller-scoped, not agent-scoped
    repeated string required_permissions = 2;
    bool audit_logging_required = 3;
    int32 rate_limit_per_minute = 4;
}

message PerformanceCharacteristics {
    int32 p50_latency_ms = 1;
    int32 p99_latency_ms = 2;
    TimeoutClass timeout_class = 3; // FAST (<100ms), MEDIUM (<1s), SLOW (<30s), LONG_RUNNING
    int32 max_result_size_bytes = 4;
}
```

### 8.3 Tool Invocation with Least Privilege and Audit Trail

```
ALGORITHM 8.1: ToolExecutor.invoke()

Input:  tool_name T, input_params P, caller_context C, deadline D
Output: ToolResult with output, metadata, and audit record

1.  // Phase 1: Discovery and validation
    tool_def ← ToolRegistry.resolve(T, version=C.required_version)
    IF tool_def = NULL:
        RETURN ToolError(code=NOT_FOUND, message="Tool not registered")
    
    validation ← JSONSchemaValidator.validate(P, tool_def.input_schema)
    IF NOT validation.valid:
        RETURN ToolError(code=INVALID_INPUT, details=validation.errors)

2.  // Phase 2: Authorization check (caller-scoped, NOT agent-scoped)
    auth_result ← AuthzEngine.check(
        caller=C.human_principal,     // Always trace to human identity
        agent=C.agent_id,
        tool=T,
        action=infer_action(P),
        permissions_required=tool_def.security.required_permissions
    )
    IF NOT auth_result.authorized:
        RETURN ToolError(code=UNAUTHORIZED, missing=auth_result.missing_permissions)

3.  // Phase 3: Mutation gate (human approval for state-changing operations)
    IF NOT tool_def.capabilities.is_read_only:
        IF tool_def.mutation_policy.requires_human_approval:
            approval ← HumanApprovalGate.request(
                action_description=format_action(T, P),
                blast_radius=estimate_blast_radius(T, P),
                timeout=min(30s, D.remaining()),
                approval_roles=tool_def.mutation_policy.approval_roles
            )
            IF NOT approval.granted:
                RETURN ToolResult(status=BLOCKED, reason="human_denied")
        
        IF tool_def.mutation_policy.supports_dry_run:
            dry_run_result ← execute_internal(T, P, dry_run=True)
            // Log dry run for audit but don't commit

4.  // Phase 4: Execution with deadline and circuit breaker
    circuit_breaker ← CircuitBreakerRegistry.get(T)
    IF circuit_breaker.state = OPEN:
        RETURN ToolError(code=SERVICE_UNAVAILABLE, retry_after=circuit_breaker.reset_time)
    
    TRY:
        result ← execute_internal(T, P, deadline=D)
        circuit_breaker.record_success()
    CATCH TimeoutException:
        circuit_breaker.record_failure()
        RETURN ToolError(code=DEADLINE_EXCEEDED)
    CATCH Exception e:
        circuit_breaker.record_failure()
        RETURN ToolError(code=INTERNAL_ERROR, message=e.message)

5.  // Phase 5: Output validation and audit
    IF tool_def.output_schema ≠ NULL:
        output_valid ← JSONSchemaValidator.validate(result, tool_def.output_schema)
        IF NOT output_valid.valid:
            RETURN ToolError(code=INVALID_OUTPUT, details="Tool returned malformed output")
    
    audit_record ← AuditLog.record({
        timestamp: now(),
        tool: T,
        caller: C.human_principal,
        agent: C.agent_id,
        input_hash: hash(P),       // Hash, not raw input, for sensitive data
        output_summary: summarize(result),
        latency_ms: elapsed(),
        status: SUCCESS
    })

6.  RETURN ToolResult(
        status=SUCCESS,
        output=result,
        metadata={latency: elapsed(), audit_id: audit_record.id}
    )
```

### 8.4 Lazy Tool Loading for Context Efficiency

Tools should not be loaded into the prompt unless they are relevant to the current task. We define a **lazy tool loading** strategy:

$$
\text{Tools}_{\text{loaded}}(q) = \{t \in \mathcal{T} \mid \text{relevance}(t, q) > \tau_{\text{tool}} \;\wedge\; |t.\text{schema}| \leq B_{\text{tool\_budget}}\}
$$

where relevance is computed by matching the query's intent graph against tool capability descriptions:

$$
\text{relevance}(t, q) = \max_{c \in t.\text{capabilities}} \cos(\text{embed}(c.\text{description}), \text{embed}(q))
$$

Only the top-$k$ most relevant tools (typically $k \leq 10$) are included in the compiled prompt, with their schemas serialized in compact form. Tools not loaded remain discoverable via a meta-tool `tool_search(query) → tool_list` that the agent can invoke when it recognizes it needs a capability not currently available.

---

## 9. End-to-End System Architecture and Production Reliability

### 9.1 Complete System Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        EXTERNAL BOUNDARY (JSON-RPC)                    │
│  ┌─────────────┐   ┌─────────────┐   ┌──────────────┐                 │
│  │  Web Client  │   │  API Client  │   │  CLI Client   │                │
│  └──────┬──────┘   └──────┬──────┘   └──────┬───────┘                 │
│         └──────────────────┼──────────────────┘                        │
│                            │ JSON-RPC 2.0 + Auth                       │
│                            ▼                                           │
│  ┌─────────────────────────────────────────────────────────────┐       │
│  │                    API GATEWAY                               │       │
│  │  Rate Limiting │ Auth │ Request Routing │ Request Shaping   │       │
│  └────────────────────────┬────────────────────────────────────┘       │
│                            │                                           │
│  ┌─────────────────────────▼────────────────────────────────────┐      │
│  │                 ORCHESTRATION LAYER                           │      │
│  │  ┌──────────────────────────────────────────────────────┐    │      │
│  │  │              SUPERVISOR AGENT                         │    │      │
│  │  │  Plan │ Decompose │ Route │ Monitor │ Merge │ Commit │    │      │
│  │  └────┬────────┬───────────┬──────────┬────────┬───────┘    │      │
│  │       │        │           │          │        │             │      │
│  │  ┌────▼───┐ ┌──▼────┐ ┌───▼───┐ ┌───▼───┐ ┌──▼─────┐      │      │
│  │  │Implemen│ │Verify │ │Retriev│ │Critic │ │Document│      │      │
│  │  │ter     │ │er     │ │er     │ │       │ │er      │      │      │
│  │  │Agent   │ │Agent  │ │Agent  │ │Agent  │ │Agent   │      │      │
│  │  └────┬───┘ └──┬────┘ └───┬───┘ └───┬───┘ └──┬─────┘      │      │
│  │       └────────┴──────────┴─────────┴────────┘             │      │
│  └──────────────────────────┬──────────────────────────────────┘      │
│                              │ gRPC (internal)                        │
│  ┌──────────────────────────▼──────────────────────────────────┐      │
│  │                 CONTEXT ENGINE                               │      │
│  │  ┌────────────┐  ┌────────────┐  ┌────────────────────┐     │      │
│  │  │  Context    │  │  Token     │  │  Positional        │     │      │
│  │  │  Compiler   │  │  Budget    │  │  Placement         │     │      │
│  │  │             │  │  Allocator │  │  Optimizer          │     │      │
│  │  └──────┬─────┘  └─────┬──────┘  └─────────┬──────────┘     │      │
│  │         └──────────────┼────────────────────┘               │      │
│  └──────────────────────────┬──────────────────────────────────┘      │
│                              │                                        │
│  ┌──────────────────────────▼──────────────────────────────────┐      │
│  │              RETRIEVAL & MEMORY LAYER                        │      │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌───────────┐   │      │
│  │  │ Vector   │  │ BM25     │  │ Knowledge│  │ Memory    │   │      │
│  │  │ Store    │  │ Index    │  │ Graph    │  │ Tiers     │   │      │
│  │  │ (HNSW)   │  │          │  │          │  │ M0..M4    │   │      │
│  │  └──────────┘  └──────────┘  └──────────┘  └───────────┘   │      │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐                  │      │
│  │  │ SQL/OLAP │  │ Live API │  │ Web      │                  │      │
│  │  │          │  │ Gateway  │  │ Search   │                  │      │
│  │  └──────────┘  └──────────┘  └──────────┘                  │      │
│  └──────────────────────────┬──────────────────────────────────┘      │
│                              │                                        │
│  ┌──────────────────────────▼──────────────────────────────────┐      │
│  │                 TOOL LAYER (MCP)                              │      │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌───────────┐   │      │
│  │  │ File     │  │ Database │  │ Browser  │  │ Code      │   │      │
│  │  │ System   │  │ Tool     │  │ Control  │  │ Execution │   │      │
│  │  │ Server   │  │ Server   │  │ Server   │  │ Server    │   │      │
│  │  └──────────┘  └──────────┘  └──────────┘  └───────────┘   │      │
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐                  │      │
│  │  │ Deploy   │  │ Monitor  │  │ Custom   │                  │      │
│  │  │ Tool     │  │ Tool     │  │ MCP Svr  │                  │      │
│  │  └──────────┘  └──────────┘  └──────────┘                  │      │
│  └─────────────────────────────────────────────────────────────┘      │
│                                                                       │
│  ┌─────────────────────────────────────────────────────────────┐      │
│  │              OBSERVABILITY & RELIABILITY                     │      │
│  │  Traces │ Metrics │ Logs │ Circuit Breakers │ Retry Budget  │      │
│  │  Cost Tracking │ Latency Histograms │ Quality Dashboards    │      │
│  └─────────────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────────────┘
```

### 9.2 Reliability Engineering

#### 9.2.1 Circuit Breaker with Exponential Backoff and Jitter

```
ALGORITHM 9.1: CircuitBreaker.execute()

Input:  operation Op, circuit_name N
Output: result or circuit-open error

STATE MACHINE:
    CLOSED → (failure_count ≥ threshold) → OPEN
    OPEN → (timeout elapsed) → HALF_OPEN
    HALF_OPEN → (probe succeeds) → CLOSED
    HALF_OPEN → (probe fails) → OPEN

1.  cb ← CircuitBreakerStore.get(N)
    
    IF cb.state = OPEN:
        IF now() < cb.next_retry_time:
            RETURN CircuitOpenError(retry_after=cb.next_retry_time - now())
        ELSE:
            cb.state ← HALF_OPEN

2.  TRY:
        result ← Op.execute()
        cb.record_success()
        IF cb.state = HALF_OPEN:
            cb.state ← CLOSED
            cb.failure_count ← 0
        RETURN result

3.  CATCH Exception:
        cb.failure_count += 1
        IF cb.failure_count ≥ cb.threshold:
            cb.state ← OPEN
            // Exponential backoff with decorrelated jitter
            base_delay ← cb.base_delay_ms * (2 ^ cb.consecutive_opens)
            jitter ← random_uniform(0, base_delay)
            cb.next_retry_time ← now() + min(base_delay + jitter, cb.max_delay_ms)
            cb.consecutive_opens += 1
        RAISE
```

#### 9.2.2 Idempotency for State-Mutating Operations

Every state-changing tool invocation and agent action carries an **idempotency key**:

$$
\text{IdempotencyKey}(op) = \text{SHA256}(\text{agent\_id} \| \text{task\_id} \| \text{operation\_type} \| \text{canonical\_params})
$$

```
ALGORITHM 9.2: IdempotentExecutor.execute()

Input:  operation Op, idempotency_key K
Output: result (from cache or fresh execution)

1.  cached ← IdempotencyStore.get(K)
    IF cached ≠ NULL AND cached.status = COMPLETED:
        RETURN cached.result    // Return cached result, no re-execution

2.  IF cached ≠ NULL AND cached.status = IN_PROGRESS:
        IF now() - cached.started_at < cached.timeout:
            RETURN ConflictError("Operation in progress")
        ELSE:
            // Stale in-progress record, allow retry
            IdempotencyStore.delete(K)

3.  // Record intent before execution
    IdempotencyStore.put(K, {status: IN_PROGRESS, started_at: now()})
    
    TRY:
        result ← Op.execute()
        IdempotencyStore.update(K, {status: COMPLETED, result: result, completed_at: now()})
        RETURN result
    CATCH Exception e:
        IdempotencyStore.update(K, {status: FAILED, error: e, failed_at: now()})
        RAISE
```

#### 9.2.3 Backpressure and Queue Isolation

The system implements **per-priority-class queues** with admission control:

$$
\text{Admit}(request) = \begin{cases}
\text{accept} & \text{if } \text{queue}[request.\text{priority}].\text{size} < \text{queue}[request.\text{priority}].\text{capacity} \\
\text{shed} & \text{if } request.\text{priority} = \text{LOW} \;\wedge\; \text{system\_load} > 0.8 \\
\text{throttle} & \text{if } \text{caller\_rate} > \text{rate\_limit}[request.\text{caller}] \\
\text{reject} & \text{otherwise}
\end{cases}
$$

Queue isolation ensures that a burst of low-priority requests cannot starve high-priority agent loop operations:

| Queue | Priority | Capacity | Timeout | Load Shed Threshold |
|-------|----------|----------|---------|---------------------|
| `agent_loop_critical` | P0 | 100 | 30s | Never shed |
| `retrieval_requests` | P1 | 500 | 10s | System load > 0.9 |
| `tool_invocations` | P1 | 200 | 30s | System load > 0.9 |
| `memory_writes` | P2 | 1000 | 60s | System load > 0.8 |
| `background_indexing` | P3 | Unbounded | 300s | System load > 0.6 |

### 9.3 Cost Optimization

#### 9.3.1 Token Cost Model

The total cost of an agent execution is:

$$
C_{\text{total}} = \sum_{i=1}^{N_{\text{calls}}} \left( c_{\text{input}} \cdot T_{\text{input}}^{(i)} + c_{\text{output}} \cdot T_{\text{output}}^{(i)} \right) + C_{\text{retrieval}} + C_{\text{tools}} + C_{\text{compute}}
$$

where $c_{\text{input}}, c_{\text{output}}$ are per-token prices for the LLM, $T_{\text{input}}^{(i)}, T_{\text{output}}^{(i)}$ are token counts for the $i$-th LLM call, and the summation runs over all LLM invocations in the agent loop.

#### 9.3.2 Cost Optimization Strategies

1. **Prompt Caching:** For stable prefixes (system policy, tool schemas), leverage provider-level prompt caching (e.g., Anthropic's prompt caching) to reduce input token costs by up to 90% on cached segments.

$$
C_{\text{cached}}^{(i)} = c_{\text{cache\_read}} \cdot T_{\text{cached}} + c_{\text{input}} \cdot T_{\text{uncached}} + c_{\text{output}} \cdot T_{\text{output}}^{(i)}
$$

where $c_{\text{cache\_read}} \ll c_{\text{input}}$.

2. **Model Routing:** Route simple subtasks to smaller, cheaper models:

$$
\text{Model}(subtask) = \begin{cases}
\mathcal{M}_{\text{large}} & \text{if complexity}(subtask) > \theta_{\text{complex}} \\
\mathcal{M}_{\text{small}} & \text{otherwise}
\end{cases}
$$

3. **Result Caching:** Cache retrieval results and tool outputs with content-addressable keys, serving repeated queries from cache.

4. **Early Termination:** If the verification gate passes on the first attempt, skip the critique-repair loop, saving 1–3 LLM calls.

### 9.4 Observability

Every boundary in the system emits structured traces following the **OpenTelemetry** specification:

```
TRACE: agent_execution
├── SPAN: query_augmentation (12ms)
│   ├── attribute: subquery_count = 3
│   └── attribute: decomposition_strategy = "RAD"
├── SPAN: retrieval (45ms)
│   ├── SPAN: vector_search (8ms)
│   │   ├── attribute: results_count = 15
│   │   └── attribute: top_score = 0.92
│   ├── SPAN: bm25_search (3ms)
│   └── SPAN: rrf_fusion (2ms)
│       └── attribute: final_results = 8
├── SPAN: context_compilation (5ms)
│   ├── attribute: total_tokens = 3847
│   ├── attribute: budget_utilization = 0.76
│   └── attribute: compression_ratio = 0.35
├── SPAN: llm_inference (1200ms)
│   ├── attribute: model = "claude-sonnet-4-20250514"
│   ├── attribute: input_tokens = 3847
│   ├── attribute: output_tokens = 512
│   ├── attribute: cache_hit_tokens = 2100
│   └── attribute: cost_usd = 0.0043
├── SPAN: verification (800ms)
│   ├── attribute: checks_passed = 4/4
│   └── attribute: confidence = 0.94
└── SPAN: memory_write (3ms)
    ├── attribute: tier = "episodic"
    └── attribute: items_promoted = 1
```

Key metrics exposed:

| Metric | Type | Description |
|--------|------|-------------|
| `agent.execution.duration_ms` | Histogram | End-to-end agent loop latency |
| `agent.loop.depth` | Counter | Number of plan-act-verify iterations |
| `agent.repair.count` | Counter | Repair attempts before success/failure |
| `context.token.utilization` | Gauge | Fraction of token budget used |
| `retrieval.latency_ms` | Histogram | Per-source retrieval latency |
| `retrieval.relevance.top_score` | Gauge | Top evidence relevance score |
| `tool.invocation.count` | Counter | Tool calls per agent execution |
| `tool.circuit_breaker.state` | Gauge | Circuit breaker state per tool |
| `memory.write.admission_rate` | Gauge | Fraction of candidates admitted |
| `cost.usd.per_execution` | Histogram | Dollar cost per agent execution |
| `hallucination.rate` | Gauge | Fraction of responses flagged ungrounded |

---

## 10. Evaluation Infrastructure and Continuous Quality Enforcement

### 10.1 Evaluation as Code

Every failure, correction, and regression is converted into a **reproducible evaluation case**:

**Definition 10.1 (Evaluation Case).** An evaluation case $\mathcal{E}$ is a tuple:

$$
\mathcal{E} = (q_{\text{input}}, \mathcal{C}_{\text{context}}, y_{\text{expected}}, \mathcal{J}_{\text{criteria}}, \mathcal{T}_{\text{trace}})
$$

where $\mathcal{J}_{\text{criteria}}$ is a set of machine-evaluable judgment functions.

### 10.2 Multi-Dimensional Judgment Functions

```
ALGORITHM 10.1: EvaluationSuite.evaluate()

Input:  model response y, evaluation case E
Output: multi-dimensional score card

1.  scores ← {}

2.  // Factual Grounding (automated)
    claims ← ClaimExtractor.extract(y)
    FOR EACH claim c IN claims:
        grounded ← EvidenceMatcher.find_support(c, E.context)
        scores["grounding"].append(grounded.score)
    scores["grounding_rate"] ← mean(scores["grounding"])

3.  // Faithfulness (automated via NLI)
    FOR EACH claim c IN claims:
        entailment ← NLIModel.check(premise=E.context, hypothesis=c)
        scores["faithfulness"].append(entailment.probability)
    scores["faithfulness_rate"] ← mean(scores["faithfulness"])

4.  // Completeness (automated)
    required_aspects ← E.criteria.required_coverage
    covered ← AspectCoverage.check(y, required_aspects)
    scores["completeness"] ← |covered| / |required_aspects|

5.  // Format Compliance (automated)
    IF E.criteria.output_schema ≠ NULL:
        compliance ← SchemaValidator.validate(y, E.criteria.output_schema)
        scores["format_compliance"] ← 1.0 IF compliance.valid ELSE 0.0

6.  // Consistency (automated, for multi-turn)
    IF E.trace.previous_responses ≠ ∅:
        contradictions ← ConsistencyChecker.find_contradictions(
            y, E.trace.previous_responses
        )
        scores["consistency"] ← 1.0 - (|contradictions| / |claims|)

7.  // LLM-as-Judge (for nuanced quality)
    judge_score ← LLMJudge.evaluate(
        query=E.input,
        response=y,
        reference=E.expected,
        rubric=E.criteria.rubric,
        model="claude-sonnet-4-20250514"
    )
    scores["judge_score"] ← judge_score

8.  // Composite Score
    scores["composite"] ← weighted_mean(scores, weights=E.criteria.dimension_weights)

9.  RETURN ScoreCard(scores, pass=scores["composite"] ≥ E.criteria.threshold)
```

### 10.3 CI/CD Integration

Evaluations execute as part of the CI/CD pipeline:

```
ALGORITHM 10.2: EvalCI.run()

Input:  code changeset ΔC, eval suite S, quality gates G
Output: pass/fail with detailed report

1.  // Stage 1: Regression Detection
    baseline_scores ← EvalStore.get_latest_baseline()
    
    FOR EACH eval_case e IN S:
        current_score ← EvaluationSuite.evaluate(model_response(e.input), e)
        delta ← current_score - baseline_scores[e.id]
        
        IF delta.composite < -G.max_regression:
            regressions.append({case: e.id, delta: delta})

2.  // Stage 2: Quality Gate Enforcement
    aggregate_scores ← aggregate(all current_scores)
    
    gates_passed ← True
    FOR EACH gate g IN G:
        IF aggregate_scores[g.metric] < g.threshold:
            gates_passed ← False
            failures.append({gate: g.name, actual: aggregate_scores[g.metric],
                            required: g.threshold})

3.  // Stage 3: Report and Decision
    report ← EvalReport(
        changeset=ΔC,
        regressions=regressions,
        gate_failures=failures,
        score_card=aggregate_scores,
        verdict=gates_passed AND |regressions| = 0
    )
    
    IF NOT report.verdict:
        CI.block_merge(report)
    ELSE:
        EvalStore.update_baseline(aggregate_scores)
        CI.allow_merge(report)

4.  RETURN report
```

### 10.4 Feedback Loop Architecture

```
Human Corrections ──┐
Failed Traces ──────┤
Reviewer Comments ──┤──→ Normalizer ──→ Eval Case Generator ──→ Eval Suite
Production Alerts ──┘                                              │
                                                                   ▼
                                                          CI/CD Pipeline
                                                                   │
                                              ┌────────────────────┤
                                              ▼                    ▼
                                     Policy Updates        Memory Updates
                                     (Prompt Compiler       (Episodic M2,
                                      adjustments)          Procedural M4)
```

Every human correction is transformed into a **regression test** that prevents the same error from recurring. This creates a **ratchet mechanism**: the system's quality can only improve over time, as the evaluation suite grows monotonically with production experience.

---

## Conclusion: Architectural Invariants

The complete Context Engineering system satisfies the following architectural invariants, each of which is mechanically enforceable:

| Invariant | Enforcement Mechanism |
|-----------|----------------------|
| No ungrounded claims in output | Provenance-tagged retrieval + faithfulness eval gate |
| Token budget never exceeded | Context compiler with hard limit enforcement |
| Agent loop always terminates | Bounded depth, time, cost, and repair counters |
| Tool mutations are human-auditable | Audit log + approval gates + caller-scoped auth |
| Memory never admits duplicates | Content hash + semantic similarity dedup on write |
| No zombie agent writes | Fence tokens on all leased task operations |
| Retrieval latency bounded | Deadline propagation + circuit breakers + fallback sources |
| Cost bounded per execution | Per-execution cost accumulator with hard ceiling |
| Quality never regresses | CI eval suite with monotonic quality ratchet |
| System observable at every boundary | OpenTelemetry traces + structured metrics + cost tracking |

This architecture transforms an LLM from an isolated, stateless text predictor into a **reliable, grounded, cost-controlled, observable, and continuously improving production system**. The critical insight is that **Context Engineering is not prompt engineering**—it is the discipline of designing the complete computational infrastructure that determines what the model sees, when it sees it, how it acts, and how correctness is verified, at every step of every execution.

---

*This report defines the reference architecture. Every equation, algorithm, and invariant is implementation-ready and maps directly to typed interfaces in the production protocol stack.*



# Context Engineering: A Principal-Level Technical Report

## Architecting Production-Grade Intelligence Systems Around Large Language Models

---

## Abstract

This report presents a rigorous, end-to-end technical treatment of **Context Engineering** — the discipline of designing typed, bounded, and verifiable information architectures that supply Large Language Models (LLMs) with precisely the right information, at precisely the right time, within precisely the right token budget. We formalize each constituent subsystem — Prompting Techniques, Query Augmentation, Retrieval, Memory, Tools, and Agents — as components of a typed protocol stack with explicit contracts, provenance guarantees, and measurable quality gates. Every concept is accompanied by formal definitions, mathematical formulations, and pseudocode algorithms suitable for implementation by research engineers and production teams operating at enterprise scale.

---

## Table of Contents

1. [Foundational Problem Statement and Formal Motivation](#1-foundational-problem-statement-and-formal-motivation)
2. [Context Engineering: Formal Definition and System Architecture](#2-context-engineering-formal-definition-and-system-architecture)
3. [Prompting Techniques: Compiled Runtime Artifacts](#3-prompting-techniques-compiled-runtime-artifacts)
4. [Query Augmentation: Deterministic Intent Decomposition and Rewriting](#4-query-augmentation-deterministic-intent-decomposition-and-rewriting)
5. [Retrieval: Hybrid Provenance-Tagged Evidence Engines](#5-retrieval-hybrid-provenance-tagged-evidence-engines)
6. [Memory: Stratified Durable State with Hard Memory Walls](#6-memory-stratified-durable-state-with-hard-memory-walls)
7. [Tools: First-Class Typed Infrastructure with Governed Mutation](#7-tools-first-class-typed-infrastructure-with-governed-mutation)
8. [Agents: Bounded Control Systems with Verification Loops](#8-agents-bounded-control-systems-with-verification-loops)
9. [Integrated System Architecture and Data Flow](#9-integrated-system-architecture-and-data-flow)
10. [Operational Constraints: Reliability, Cost, Latency, and Observability](#10-operational-constraints-reliability-cost-latency-and-observability)
11. [Conclusion](#11-conclusion)

---

## 1. Foundational Problem Statement and Formal Motivation

### 1.1 The Isolation Theorem

**Definition 1.1 (LLM as a Bounded Stateless Function).** A Large Language Model $\mathcal{M}$ is a parameterized conditional distribution:

$$\mathcal{M}_\theta : \mathcal{V}^{\leq W} \rightarrow \Delta(\mathcal{V})$$

where $\mathcal{V}$ is the token vocabulary, $W \in \mathbb{N}$ is the **context window bound** (measured in tokens), $\theta$ are frozen parameters at inference time, and $\Delta(\mathcal{V})$ denotes the probability simplex over $\mathcal{V}$.

The model autoregressively generates token $t_{n+1}$ conditioned on $\mathbf{t}_{1:n}$:

$$P(t_{n+1} \mid \mathbf{t}_{1:n}; \theta) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V \bigg|_{\text{position}\ n+1}$$

subject to the **hard constraint** $n < W$.

**Theorem 1.1 (Fundamental Isolation).** Given a model $\mathcal{M}_\theta$ with training data cutoff at timestamp $T_{\text{cut}}$ and context window $W$:

1. **Temporal Isolation:** $\forall\ \text{event}\ e$ where $\text{timestamp}(e) > T_{\text{cut}}$, $\mathcal{M}_\theta$ assigns probability $P(e \mid \theta) \approx P_{\text{prior}}(e)$ — the model's output degenerates to its prior, yielding hallucinated or vacuous responses.

2. **Data Isolation:** $\forall\ \text{document}\ d \notin \mathcal{D}_{\text{train}}$, the model has zero grounding: $I(d; \theta) = 0$ where $I$ denotes mutual information between the document content and the model parameters.

3. **State Isolation:** Between independent inference calls at times $t_i$ and $t_j$ ($i \neq j$), the model retains no state: $\mathcal{M}_\theta(\cdot \mid \mathbf{t}^{(t_i)}) \perp\!\!\!\perp \mathcal{M}_\theta(\cdot \mid \mathbf{t}^{(t_j)})$.

4. **Context Truncation:** When the accumulated token sequence $|\mathbf{t}| > W$, information must be evicted. Under naive FIFO truncation, the expected information loss is:

$$\mathcal{L}_{\text{trunc}} = \sum_{k=1}^{|\mathbf{t}| - W} \text{Utility}(t_k) \cdot \mathbb{1}[t_k\ \text{evicted}]$$

which is **unbounded** and **uncontrolled** without explicit management.

**Corollary 1.1.** No improvement to $\theta$ (model training, fine-tuning, RLHF) resolves these isolation properties. They are **architectural invariants** of the transformer inference regime. Resolution requires an **external system** that manages context construction.

### 1.2 The Hallucination Mechanism: A Formal Characterization

Hallucination is not a bug — it is the **expected behavior** of a generative model operating beyond its grounding boundary.

**Definition 1.2 (Grounding Boundary).** For a given query $q$ and a context $C$ present in the context window, the grounding boundary $\mathcal{G}(q, C, \theta)$ is the set of claims the model can produce that are **entailed** by $C$ or reliably encoded in $\theta$:

$$\mathcal{G}(q, C, \theta) = \{c \mid C \models c\} \cup \{c \mid P(c \mid \theta) > \tau_{\text{reliable}}\}$$

**Definition 1.3 (Hallucination).** A generated claim $\hat{c}$ is a hallucination if:

$$\hat{c} \notin \mathcal{G}(q, C, \theta) \quad \wedge \quad P(\hat{c} \mid q, C; \theta) > \epsilon$$

That is, the model assigns non-negligible probability to a claim not entailed by its context or reliably encoded in its parameters.

**Proposition 1.1 (Hallucination Rate and Context Quality).** The expected hallucination rate $\mathbb{E}[\mathcal{H}]$ is inversely related to context quality:

$$\mathbb{E}[\mathcal{H}] \propto \frac{1}{\text{Coverage}(C, q) \cdot \text{Precision}(C, q) \cdot \text{Authority}(C)}$$

where:
- $\text{Coverage}(C, q) = \frac{|\text{InfoNeeds}(q) \cap \text{InfoProvided}(C)|}{|\text{InfoNeeds}(q)|}$
- $\text{Precision}(C, q) = \frac{|\text{RelevantTokens}(C, q)|}{|C|}$
- $\text{Authority}(C) = \min_{d \in \text{sources}(C)} \text{TrustScore}(d)$

**Implication:** Hallucination control is a **context engineering** problem, not a model-training problem. The system must maximize coverage, precision, and authority of the context delivered to the model.

### 1.3 The Context Window as a Computational Resource

The context window $W$ is not merely a limitation — it is the **primary computational resource** of an LLM-based system. Every token consumed by instructions, retrieved evidence, tool schemas, memory summaries, or conversation history is a token **not available** for reasoning, generation, or nuanced analysis.

**Definition 1.4 (Token Budget Allocation).** A context engineering system partitions $W$ into disjoint allocations:

$$W = B_{\text{sys}} + B_{\text{task}} + B_{\text{retrieval}} + B_{\text{tools}} + B_{\text{memory}} + B_{\text{history}} + B_{\text{reasoning}} + B_{\text{output}}$$

where:
- $B_{\text{sys}}$: System prompt / role policy
- $B_{\text{task}}$: Task-specific instructions and constraints
- $B_{\text{retrieval}}$: Retrieved evidence payloads
- $B_{\text{tools}}$: Tool schemas and affordances
- $B_{\text{memory}}$: Memory summaries (episodic, semantic, procedural)
- $B_{\text{history}}$: Conversation history / session state
- $B_{\text{reasoning}}$: Reserved for chain-of-thought / planning
- $B_{\text{output}}$: Reserved for generation

**Optimization Objective:**

$$\max_{\text{allocation}} \quad \text{TaskUtility}(\mathcal{M}_\theta(\text{Context}(W))) \quad \text{s.t.} \quad \sum B_i \leq W$$

This is a **constrained optimization problem** that Context Engineering solves at system design time and at runtime for every inference call.

---

## 2. Context Engineering: Formal Definition and System Architecture

### 2.1 Formal Definition

**Definition 2.1 (Context Engineering).** Context Engineering is the discipline of designing, constructing, and operating the **deterministic information pipeline** $\Pi$ that, for every inference invocation, assembles a bounded context $C$ from heterogeneous sources and delivers it to $\mathcal{M}_\theta$ such that:

$$C = \Pi(q, \mathcal{S}, \mathcal{T}, \mathcal{M}_{\text{mem}}, \mathcal{H}_{\text{session}}, \mathcal{P}, W)$$

where:
- $q$: User query or task specification
- $\mathcal{S}$: Available data sources (document stores, databases, APIs, live streams)
- $\mathcal{T}$: Available tool schemas and affordances
- $\mathcal{M}_{\text{mem}}$: Multi-tier memory system
- $\mathcal{H}_{\text{session}}$: Session history and working state
- $\mathcal{P}$: System policies and role constraints
- $W$: Token budget

and $\Pi$ satisfies the following invariants:

1. **Boundedness:** $|C| \leq W$
2. **Determinism:** Given identical inputs, $\Pi$ produces identical $C$ (modulo controlled randomness in sampling)
3. **Provenance:** Every element $c_i \in C$ carries a provenance tag $\text{prov}(c_i) = (\text{source}, \text{timestamp}, \text{authority}, \text{method})$
4. **Relevance:** $\text{Precision}(C, q) > \tau_{\text{relevance}}$
5. **Freshness:** $\max_{c_i \in C}(\text{now} - \text{timestamp}(c_i)) < \delta_{\text{staleness}}$ where applicable
6. **Completeness:** $\text{Coverage}(C, q) > \tau_{\text{coverage}}$

### 2.2 System Architecture Overview

The Context Engineering platform is organized as a **typed protocol stack** with six core subsystems. Each subsystem has well-defined contracts, boundaries, and failure modes.

```
┌──────────────────────────────────────────────────────────────────┐
│                    APPLICATION / USER BOUNDARY                   │
│                    Protocol: JSON-RPC 2.0                        │
│    (Typed request/response, pagination, deadlines, error codes)  │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌──────────┐  ┌──────────────┐  ┌───────────┐  ┌────────────┐ │
│  │  QUERY   │  │  RETRIEVAL   │  │  MEMORY   │  │   TOOLS    │ │
│  │  AUGMENT │→│   ENGINE     │→│  MANAGER  │→│  REGISTRY  │ │
│  │          │  │              │  │           │  │            │ │
│  │ Decompose│  │ Hybrid Search│  │ 5-Tier    │  │ MCP/gRPC   │ │
│  │ Rewrite  │  │ Provenance   │  │ Hierarchy │  │ Typed I/O  │ │
│  │ Route    │  │ Rank & Fuse  │  │ Hard Wall │  │ Least Priv │ │
│  └──────────┘  └──────────────┘  └───────────┘  └────────────┘ │
│         │              │               │               │        │
│         └──────────────┴───────────────┴───────────────┘        │
│                              │                                   │
│                    ┌─────────▼─────────┐                        │
│                    │  PREFILL COMPILER  │                        │
│                    │  (Context Assembly)│                        │
│                    │  Token Budgeting   │                        │
│                    │  Deterministic     │                        │
│                    └─────────┬─────────┘                        │
│                              │                                   │
│                    ┌─────────▼─────────┐                        │
│                    │   AGENT LOOP      │                        │
│                    │  Plan→Act→Verify  │                        │
│                    │  →Critique→Repair │                        │
│                    │  →Commit          │                        │
│                    └───────────────────┘                        │
│                                                                  │
├──────────────────────────────────────────────────────────────────┤
│              INTERNAL SERVICE MESH: gRPC / Protobuf              │
│     (Low-latency, typed, streaming, deadline propagation)        │
├──────────────────────────────────────────────────────────────────┤
│              TOOL / RESOURCE LAYER: MCP Protocol                 │
│     (Discovery, schema, pagination, change notifications)        │
└──────────────────────────────────────────────────────────────────┘
```

**Data Flow Sequence (Numbered to Match the Original Diagram):**

| Step | Component | Operation |
|------|-----------|-----------|
| 1 | User → Agent | User submits query via JSON-RPC |
| 2 | Agent → Query Augmentation | Decompose, rewrite, route |
| 3 | Query Augmentation → Retrieval (RAG + Vector Search) | Hybrid multi-source retrieval |
| 4 | Retrieval → Prefill Compiler | Provenance-tagged evidence ranked and fused |
| 5 | Agent → Tools (MCP / Databases / Actions) | Tool invocation for live data or mutations |
| 6 | Tools → Agent | Structured tool responses |
| 7 | Agent → Memory (Short-Term + Long-Term) | Store validated context, read priors |
| 8 | Prefill Compiler → LLM → User | Compiled context → Generation → Response |

---

## 3. Prompting Techniques: Compiled Runtime Artifacts

### 3.1 Motivation: Beyond Artisanal Prompt Craft

Prompting in a production agentic system is **not** a creative writing exercise. It is the construction of a **compiled runtime artifact** — a deterministic, version-controlled, token-budgeted preamble that configures the LLM's inference behavior as precisely as a function signature configures a compiler.

**Definition 3.1 (Prompt Artifact).** A prompt artifact $\mathcal{A}$ is a structured composition:

$$\mathcal{A} = \bigoplus_{i=1}^{k} \text{Section}_i \quad \text{where} \quad \text{Section}_i = (\text{role}_i, \text{content}_i, \text{priority}_i, \text{budget}_i)$$

subject to $\sum_{i} \text{budget}_i \leq B_{\text{sys}} + B_{\text{task}}$.

### 3.2 Prompt Compilation Pipeline

The **Prefill Compiler** assembles the prompt artifact at runtime, not at authoring time. This is analogous to a just-in-time compiler that emits machine code from intermediate representations.

**Algorithm 3.1: Prefill Compiler**

```
ALGORITHM PrefillCompiler(
    policy: RolePolicy,
    task: TaskObjective,
    tools: List[ToolSchema],
    evidence: List[ProvenanceTaggedChunk],
    memory: MemorySummary,
    history: CompressedSessionHistory,
    W: int  // total token budget
) → CompiledContext

INPUT VALIDATION:
    ASSERT policy.version ∈ SUPPORTED_VERSIONS
    ASSERT |tools| ≤ MAX_TOOL_SLOTS

// Phase 1: Budget Allocation
B_sys    ← STATIC_BUDGET(policy)                    // Fixed: role, constraints
B_task   ← MEASURE_TOKENS(task)                     // Measured: task description
B_tools  ← Σ MEASURE_TOKENS(t.schema) for t ∈ tools  // Lazy: only active tools
B_output ← RESERVE_OUTPUT_BUDGET(task.expected_length)
B_reason ← RESERVE_REASONING_BUDGET(task.complexity)
B_avail  ← W - B_sys - B_task - B_tools - B_output - B_reason

// Phase 2: Priority-Ranked Allocation of Variable Context
memory_payload  ← memory.serialize(max_tokens = 0.15 * B_avail)
history_payload ← history.compress(max_tokens = 0.20 * B_avail)
evidence_sorted ← RANK(evidence, by=[authority, relevance, freshness])
evidence_payload ← GREEDILY_PACK(evidence_sorted,
                                   max_tokens = B_avail - |memory_payload| - |history_payload|)

// Phase 3: Deterministic Assembly
context ← CONCATENATE(
    SECTION("system", policy.render()),
    SECTION("task", task.render()),
    SECTION("memory", memory_payload),
    SECTION("evidence", evidence_payload,
            with_provenance_tags = TRUE),
    SECTION("tools", [t.schema for t in tools]),
    SECTION("history", history_payload),
    SECTION("instructions", "Answer based ONLY on the provided evidence. "
                           "Cite sources by provenance ID. "
                           "If evidence is insufficient, state explicitly.")
)

// Phase 4: Invariant Verification
ASSERT |context| ≤ W - B_output - B_reason
ASSERT PROVENANCE_COVERAGE(evidence_payload) == 1.0  // All chunks tagged
ASSERT NO_DUPLICATE_SECTIONS(context)

RETURN CompiledContext(
    payload = context,
    token_count = |context|,
    budget_remaining = W - |context|,
    trace_id = GENERATE_TRACE_ID(),
    compilation_timestamp = NOW()
)
```

### 3.3 Structured Prompting Taxonomy (SOTA Techniques)

We formalize the principal-level prompting techniques as **inference-time compute strategies** rather than stylistic choices.

#### 3.3.1 Chain-of-Thought with Verified Decomposition (CoT-VD)

Standard Chain-of-Thought (Wei et al., 2022) elicits intermediate reasoning steps. At SOTA level, we augment with **verification at each decomposition step**.

**Formal Specification:**

Let $q$ be a complex query decomposable into sub-problems $\{s_1, \ldots, s_n\}$.

$$\text{CoT-VD}(q) = \bigcirc_{i=1}^{n} \left[ \text{Solve}(s_i) \rightarrow \text{Verify}(s_i, \text{evidence}) \rightarrow \text{Commit}(s_i) \mid \text{Repair}(s_i) \right]$$

where $\bigcirc$ denotes sequential composition, and each step proceeds only if verification passes.

**Algorithm 3.2: Verified Chain-of-Thought**

```
ALGORITHM CoT_VerifiedDecomposition(q, evidence, max_steps)
    steps ← DECOMPOSE(q)  // LLM-generated decomposition
    results ← []
    
    FOR i = 1 TO |steps|:
        IF i > max_steps: ABORT("Recursion bound exceeded")
        
        s_i ← steps[i]
        answer_i ← LLM.solve(s_i, context = evidence + results)
        
        // Verification gate
        verification ← VERIFY(answer_i, against = evidence)
        IF verification.status == SUPPORTED:
            results.APPEND((s_i, answer_i, verification.citations))
        ELIF verification.status == CONTRADICTED:
            repaired ← LLM.repair(s_i, answer_i, verification.contradiction_evidence)
            results.APPEND((s_i, repaired, verification.citations))
        ELIF verification.status == UNSUPPORTED:
            results.APPEND((s_i, "INSUFFICIENT_EVIDENCE", ∅))
    
    final_answer ← LLM.synthesize(results, original_query = q)
    RETURN (final_answer, results)  // Full trace for observability
```

#### 3.3.2 Constrained Decoding via Structured Output Schemas

Rather than hoping the model produces valid JSON/structured output, SOTA systems enforce it through **grammar-constrained decoding** or **tool-call function schemas**.

**Formal Definition:**

$$P_{\text{constrained}}(t_{n+1} \mid \mathbf{t}_{1:n}) = \frac{P(t_{n+1} \mid \mathbf{t}_{1:n}) \cdot \mathbb{1}[t_{n+1} \in \mathcal{G}_n]}{\sum_{v \in \mathcal{G}_n} P(v \mid \mathbf{t}_{1:n})}$$

where $\mathcal{G}_n \subseteq \mathcal{V}$ is the set of tokens **valid** at position $n$ according to the output grammar (e.g., JSON schema, Protobuf message structure).

This eliminates an entire class of failures (malformed outputs, missing fields, type errors) **mechanically** rather than through prompt instructions.

#### 3.3.3 Self-Consistency with Majority Voting and Provenance Weighting

For high-stakes queries, generate $k$ independent reasoning chains and select the answer with highest weighted agreement:

$$\hat{a} = \arg\max_{a} \sum_{i=1}^{k} w_i \cdot \mathbb{1}[\text{answer}_i = a]$$

where weights $w_i$ are derived from:

$$w_i = \alpha \cdot \text{CitationCount}(i) + \beta \cdot \text{ReasoningDepth}(i) + \gamma \cdot \text{InternalConsistency}(i)$$

with $\alpha + \beta + \gamma = 1$. This is superior to unweighted majority voting because it privileges **well-grounded** reasoning chains over confident but unsubstantiated ones.

#### 3.3.4 Prompt Versioning and A/B Evaluation

Every prompt artifact is treated as a **versioned deployment artifact**:

```protobuf
message PromptArtifact {
    string artifact_id = 1;
    string version = 2;          // semver: major.minor.patch
    string role_policy = 3;
    repeated Section sections = 4;
    TokenBudget budget = 5;
    string hash = 6;             // SHA-256 of rendered content
    Timestamp created_at = 7;
    repeated EvalResult eval_history = 8;  // Linked quality gates
}
```

**Quality Gate:** No prompt version is promoted to production without passing a minimum **eval score** on the task-specific benchmark suite (see Section 9).

### 3.4 Anti-Pattern Catalogue

| Anti-Pattern | Failure Mode | SOTA Replacement |
|---|---|---|
| Unbudgeted context stuffing | Reasoning degradation, "lost in the middle" effect | Token-budgeted prefill compilation |
| Stylistic verbosity in system prompts | Wasted tokens, reduced reasoning capacity | Compressed constraint lists with explicit priorities |
| Reliance on "please" / "be careful" phrasing | No mechanical enforcement | Constrained decoding + verification loops |
| Copy-paste prompt engineering | Version drift, untested changes | Versioned artifacts with CI/CD eval gates |
| Single monolithic prompt | Inflexible, impossible to A/B test components | Compositional section-based assembly |

---

## 4. Query Augmentation: Deterministic Intent Decomposition and Rewriting

### 4.1 The Problem of Naive Query Submission

A user's raw query $q_{\text{raw}}$ is typically:
- **Ambiguous:** multiple valid interpretations
- **Incomplete:** missing constraints, temporal scope, entity resolution
- **Monolithic:** a single query when multiple retrieval sources require different formulations
- **Vocabulary-mismatched:** user terminology ≠ corpus terminology

Issuing $q_{\text{raw}}$ directly to a retrieval system yields recall and precision losses that propagate through the entire pipeline.

**Proposition 4.1 (Query-Retrieval Mismatch Loss).** Let $\mathcal{R}^*$ be the ideal retrieval set and $\mathcal{R}(q_{\text{raw}})$ be the actual retrieval set from a naive query. The information loss is:

$$\mathcal{L}_{\text{mismatch}} = 1 - \frac{|\mathcal{R}(q_{\text{raw}}) \cap \mathcal{R}^*|}{|\mathcal{R}^*|} = 1 - \text{Recall}(q_{\text{raw}})$$

In production corpora, $\mathcal{L}_{\text{mismatch}}$ typically ranges from 0.3 to 0.7, meaning 30–70% of relevant information is **never retrieved**.

### 4.2 Query Augmentation Pipeline

Query Augmentation is a **deterministic, multi-stage transformation pipeline** that converts $q_{\text{raw}}$ into a set of optimized, routed sub-queries.

**Definition 4.1 (Query Augmentation Function):**

$$\text{QA}: q_{\text{raw}} \times \mathcal{M}_{\text{mem}} \times \mathcal{H}_{\text{session}} \rightarrow \{(q_i, r_i, \tau_i, f_i)\}_{i=1}^{m}$$

where each tuple contains:
- $q_i$: Rewritten sub-query optimized for source $r_i$
- $r_i$: Target retrieval source / index
- $\tau_i$: Latency tier classification (hot / warm / cold)
- $f_i$: Required filters (temporal, metadata, access control)

### 4.3 Stage 1: Intent Classification and Decomposition

**Algorithm 4.1: Intent-Aware Decomposition**

```
ALGORITHM IntentDecompose(q_raw, session_context, memory)
    // Step 1: Coreference Resolution
    q_resolved ← RESOLVE_COREFERENCES(q_raw, session_context)
    // e.g., "What about their Q3 numbers?" →
    //       "What are Acme Corp's Q3 2024 revenue numbers?"
    
    // Step 2: Intent Classification
    intent ← CLASSIFY_INTENT(q_resolved)
    // Taxonomy: {FACTUAL_LOOKUP, ANALYTICAL, COMPARATIVE,
    //            PROCEDURAL, CREATIVE, MULTI_HOP, TEMPORAL}
    
    // Step 3: Information Need Decomposition
    IF intent.type == MULTI_HOP:
        sub_queries ← MULTI_HOP_DECOMPOSE(q_resolved)
        // "Compare Acme's Q3 to their Q2 and to Beta Corp's Q3"
        // → ["Acme Corp Q3 2024 revenue",
        //    "Acme Corp Q2 2024 revenue",
        //    "Beta Corp Q3 2024 revenue"]
    ELIF intent.type == TEMPORAL:
        sub_queries ← TEMPORAL_DECOMPOSE(q_resolved, memory.temporal_context)
    ELSE:
        sub_queries ← [q_resolved]
    
    // Step 4: Constraint Extraction
    FOR EACH sq IN sub_queries:
        sq.constraints ← EXTRACT_CONSTRAINTS(sq)
        // {entity: "Acme Corp", metric: "revenue",
        //  period: "Q3 2024", unit: "USD"}
    
    RETURN sub_queries
```

### 4.4 Stage 2: Query Rewriting and Expansion

For each sub-query, generate retrieval-optimized reformulations.

**Technique 4.1: Hypothetical Document Embedding (HyDE) with Typed Constraints**

Rather than searching with the query directly, generate a **hypothetical ideal answer** and use its embedding as the search vector.

$$\text{HyDE}(q_i) = \text{Embed}\!\left(\mathcal{M}_\theta\!\left(\text{"Write a short passage that would perfectly answer: } q_i\text{"}\right)\right)$$

**Mathematical justification:** The hypothetical answer $\hat{a}_i$ lives in the same embedding subspace as actual answer passages, reducing the query-document embedding gap:

$$\text{sim}(\text{Embed}(\hat{a}_i), \text{Embed}(d^*)) > \text{sim}(\text{Embed}(q_i), \text{Embed}(d^*))$$

where $d^*$ is the ideal retrieved document.

**Technique 4.2: Multi-Representation Query Expansion**

Generate $k$ semantically diverse reformulations to maximize recall across embedding space regions:

```
ALGORITHM MultiRepresentationExpansion(q_i, k=3)
    expansions ← []
    
    // Synonym/terminology expansion
    expansions.ADD(REWRITE(q_i, style="technical_terminology"))
    
    // Hypothetical answer embedding  
    expansions.ADD(HyDE(q_i))
    
    // Step-back abstraction (for overly specific queries)
    expansions.ADD(STEP_BACK(q_i))
    // "Acme Q3 2024 revenue" → "Acme Corp financial performance 2024"
    
    // Keyword extraction for exact-match augmentation
    keywords ← EXTRACT_KEYWORDS(q_i)  // BM25-optimized terms
    expansions.ADD(keywords)
    
    RETURN expansions[:k]
```

### 4.5 Stage 3: Source Routing

Each sub-query is routed to the appropriate retrieval source based on **schema matching**, **latency tier**, and **authority requirements**.

**Algorithm 4.2: Query Router**

```
ALGORITHM RouteQueries(sub_queries, source_registry)
    routed ← []
    
    FOR EACH sq IN sub_queries:
        candidates ← source_registry.MATCH(
            entity_type = sq.constraints.entity_type,
            data_domain = sq.constraints.domain,
            freshness_required = sq.constraints.temporal_bound,
            authority_minimum = sq.constraints.authority_threshold
        )
        
        // Latency-tier assignment
        FOR EACH source IN candidates:
            IF source.p99_latency < 50ms:
                tier ← HOT    // Cache, in-memory index
            ELIF source.p99_latency < 500ms:
                tier ← WARM   // Vector DB, search index
            ELSE:
                tier ← COLD   // External API, database join
        
        // Select top sources by (relevance × 1/latency × authority)
        selected ← TOP_K(candidates, k=2,
                         score = λs: s.relevance * s.authority / s.latency)
        
        FOR EACH source IN selected:
            routed.ADD(RoutedQuery(
                query = sq,
                source = source,
                tier = tier,
                timeout = source.sla_timeout,
                filters = sq.constraints.to_filter_expression()
            ))
    
    RETURN routed
```

### 4.6 Query Augmentation Quality Metrics

| Metric | Formula | Target |
|---|---|---|
| Decomposition Completeness | $\frac{\text{InfoNeeds covered by sub-queries}}{\text{Total InfoNeeds in } q_{\text{raw}}}$ | $\geq 0.95$ |
| Routing Precision | $\frac{\text{Queries routed to correct source}}{\text{Total routed queries}}$ | $\geq 0.90$ |
| Expansion Diversity | $1 - \frac{1}{\binom{k}{2}} \sum_{i<j} \text{sim}(e_i, e_j)$ | $\geq 0.40$ |
| Latency Overhead | $\text{Time(QA pipeline)} / \text{Time(naive query)}$ | $\leq 1.5\times$ |

---

## 5. Retrieval: Hybrid Provenance-Tagged Evidence Engines

### 5.1 Retrieval as Deterministic Evidence Production

Retrieval in a Context Engineering system is **not** ad hoc RAG (retrieve-and-append). It is a **deterministic evidence production engine** that:

1. Accepts routed queries from the Query Augmentation layer
2. Executes hybrid retrieval across multiple modalities
3. Ranks results by a composite scoring function
4. Attaches provenance metadata to every returned chunk
5. Returns **evidence**, not anonymous context blobs

**Definition 5.1 (Evidence Chunk).** An evidence chunk $e$ is a typed record:

```
Evidence {
    content: string,                    // The actual text
    provenance: {
        source_id: string,             // Document/table/API identifier
        source_type: enum,             // DOCUMENT | DATABASE | API | MEMORY
        chunk_id: string,              // Unique within source
        version: string,               // Source version at retrieval time
        timestamp: datetime,           // Source last modified
        authority: float [0,1],        // Trust score
        retrieval_method: enum,        // EXACT | SEMANTIC | GRAPH | HYBRID
        lineage: List[TransformOp]     // How this chunk was derived
    },
    score: float,                      // Composite relevance score
    token_count: int                   // Pre-measured for budget allocation
}
```

### 5.2 Hybrid Retrieval Architecture

SOTA retrieval combines multiple retrieval modalities, each with complementary strengths:

**Definition 5.2 (Hybrid Retrieval Function):**

$$\text{Retrieve}(q_i) = \text{RankFuse}\!\left(\bigcup_{m \in \mathcal{M}_{\text{retrieval}}} m(q_i)\right)$$

where $\mathcal{M}_{\text{retrieval}} = \{\text{ExactMatch}, \text{SemanticSearch}, \text{GraphTraversal}, \text{MetadataFilter}, \text{TemporalIndex}\}$.

#### 5.2.1 Exact Match (Sparse Retrieval)

**Algorithm:** BM25 with corpus-specific tuning.

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

**SOTA Enhancement — Learned Sparse Representations (SPLADE v2):**

Instead of raw term frequency, use a trained sparse encoder $\phi_{\text{sparse}}$ that produces a sparse vector where each dimension corresponds to a vocabulary term, but weights are **learned** to maximize retrieval effectiveness:

$$\mathbf{s}(d) = \max_{\text{positions}} \log(1 + \text{ReLU}(\mathbf{W} \cdot \text{BERT}(d)))$$

This captures term importance beyond raw frequency while maintaining inverted-index-compatible sparse structure.

#### 5.2.2 Semantic Search (Dense Retrieval)

**Algorithm:** Dual-encoder approximate nearest neighbor search.

$$\text{SemanticScore}(q, d) = \text{sim}(\phi_q(q), \phi_d(d))$$

where $\phi_q$ and $\phi_d$ are query and document encoders (e.g., E5-large-v2, BGE-M3, or domain-fine-tuned encoders), and $\text{sim}$ is cosine similarity.

**SOTA Enhancement — Late Interaction (ColBERT v2):**

Rather than a single vector per query/document, use **token-level interaction**:

$$\text{ColBERTScore}(q, d) = \sum_{i=1}^{|q|} \max_{j=1}^{|d|} \mathbf{q}_i^\top \mathbf{d}_j$$

This preserves fine-grained matching while remaining efficient through offline document encoding and runtime MaxSim operations.

**Complexity:** $O(|q| \cdot |d|)$ per candidate, but only applied to top-$k$ candidates from ANN pre-filtering.

#### 5.2.3 Graph-Based Contextual Retrieval

For queries requiring relational or multi-hop reasoning, traverse a **knowledge graph** or **chunk graph** where edges represent semantic, referential, or structural relationships.

**Algorithm 5.1: Graph-Augmented Retrieval**

```
ALGORITHM GraphAugmentedRetrieval(seed_chunks, graph, max_hops=2)
    frontier ← seed_chunks  // Initial retrieval results
    visited ← SET(seed_chunks)
    augmented ← LIST(seed_chunks)
    
    FOR hop = 1 TO max_hops:
        next_frontier ← []
        FOR EACH chunk IN frontier:
            neighbors ← graph.ADJACENT(chunk,
                         edge_types=[REFERENCES, CONTINUES, DEFINES, CONTRADICTS])
            FOR EACH (neighbor, edge_type, edge_weight) IN neighbors:
                IF neighbor ∉ visited AND edge_weight > τ_edge:
                    augmented.ADD(neighbor,
                                  provenance.method = "GRAPH_HOP_" + str(hop))
                    next_frontier.ADD(neighbor)
                    visited.ADD(neighbor)
        frontier ← next_frontier
    
    RETURN augmented
```

**Use Cases:**
- Retrieving the definition of a term referenced in a retrieved passage
- Following citation chains for authority verification
- Connecting related regulatory clauses across documents

#### 5.2.4 Metadata-Filtered Retrieval

Apply **pre-retrieval filtering** using structured metadata to reduce the search space before semantic scoring.

$$\text{FilteredSet}(q_i) = \{d \in \mathcal{D} \mid d.\text{metadata} \models f_i\}$$

where $f_i$ is the filter expression from the routed query (e.g., `department = "engineering" AND date ≥ "2024-01-01" AND access_level ≤ user.clearance`).

**Implementation:** Hybrid vector databases (Qdrant, Weaviate, Milvus) support pre-filtering at the index level, avoiding post-hoc filtering that wastes retrieval budget.

### 5.3 Rank Fusion

Given result sets from multiple retrieval modalities, fuse into a single ranked list.

**SOTA Technique: Reciprocal Rank Fusion (RRF) with Authority Weighting:**

$$\text{RRF}(d) = \sum_{m \in \mathcal{M}} \frac{w_m}{k + \text{rank}_m(d)} \cdot \text{Authority}(d)$$

where $k = 60$ (standard constant), $w_m$ is the weight assigned to modality $m$, and $\text{Authority}(d)$ is the source trust score.

**Algorithm 5.2: Weighted RRF with Provenance**

```
ALGORITHM WeightedRRF(result_sets, weights, k=60)
    scores ← DefaultDict(float)
    provenance_map ← DefaultDict(list)
    
    FOR EACH (results, weight) IN ZIP(result_sets, weights):
        FOR rank, doc IN ENUMERATE(results, start=1):
            rrf_contribution ← weight / (k + rank)
            authority_factor ← doc.provenance.authority
            scores[doc.id] += rrf_contribution * authority_factor
            provenance_map[doc.id].ADD(doc.provenance)
    
    ranked ← SORT(scores.items(), by=value, descending=TRUE)
    
    RETURN [
        Evidence(
            content = FETCH(doc_id),
            provenance = MERGE_PROVENANCE(provenance_map[doc_id]),
            score = score,
            token_count = COUNT_TOKENS(FETCH(doc_id))
        )
        FOR (doc_id, score) IN ranked
    ]
```

### 5.4 Chunking Strategies: Per-Document-Class Optimization

Chunking is not a single strategy — it is a **policy** selected per document class to optimize for retrieval precision, contextual completeness, and downstream synthesis.

**Definition 5.3 (Chunking Policy Function):**

$$\text{ChunkPolicy}: \text{DocClass} \rightarrow (\text{Strategy}, \text{ChunkSize}, \text{Overlap}, \text{Metadata})$$

| Document Class | Chunking Strategy | Chunk Size | Overlap | Rationale |
|---|---|---|---|---|
| Code files | **AST-based structural** | Function/class level | Cross-reference headers | Preserves syntactic units; avoids splitting function bodies |
| Legal/regulatory | **Hierarchical (parent-child)** | Section → Subsection → Clause | Include section header in each child | Maintains legal structure; enables upward traversal |
| Technical documentation | **Semantic (embedding-boundary)** | 512–1024 tokens | 128 tokens | Captures complete conceptual units |
| Tabular data | **Row-group with schema header** | 20–50 rows | Schema repeated per chunk | Preserves column semantics |
| Conversation logs | **Turn-pair segmentation** | User+Assistant turn pairs | 1 prior turn | Preserves dialogue coherence |
| API documentation | **Agentic (per-endpoint)** | One endpoint per chunk | Cross-references | Directly maps to tool schemas |

**Algorithm 5.3: Document-Class-Aware Chunking**

```
ALGORITHM AdaptiveChunk(document, doc_class)
    policy ← CHUNK_POLICY_REGISTRY.GET(doc_class)
    
    SWITCH policy.strategy:
        CASE AST_STRUCTURAL:
            ast ← PARSE_AST(document)
            chunks ← [node.source_text FOR node IN ast.walk()
                       IF node.type ∈ {FUNCTION, CLASS, MODULE}]
            FOR EACH chunk IN chunks:
                chunk.metadata ← {
                    "parent_class": node.parent.name,
                    "imports": node.import_dependencies,
                    "signature": node.signature
                }
        
        CASE HIERARCHICAL:
            tree ← PARSE_HIERARCHY(document)  // Heading-based tree
            chunks ← []
            FOR EACH leaf IN tree.leaves():
                context_header ← " > ".join(leaf.ancestor_titles())
                chunks.ADD(context_header + "\n" + leaf.content)
        
        CASE SEMANTIC:
            sentences ← SENTENCE_SPLIT(document)
            embeddings ← ENCODE_BATCH(sentences)
            boundaries ← DETECT_BOUNDARIES(embeddings,
                                            threshold = policy.similarity_threshold)
            chunks ← MERGE_BETWEEN_BOUNDARIES(sentences, boundaries,
                                               max_tokens = policy.chunk_size)
        
        CASE AGENTIC:
            endpoints ← PARSE_API_SPEC(document)
            chunks ← [endpoint.to_chunk() FOR endpoint IN endpoints]
    
    // Attach universal metadata
    FOR EACH chunk IN chunks:
        chunk.provenance ← {
            source_id: document.id,
            chunk_strategy: policy.strategy,
            created_at: NOW(),
            token_count: COUNT_TOKENS(chunk.content)
        }
    
    RETURN chunks
```

### 5.5 Retrieval Quality Metrics

| Metric | Definition | Target |
|---|---|---|
| Recall@k | $\frac{|\text{Relevant} \cap \text{Retrieved@k}|}{|\text{Relevant}|}$ | $\geq 0.90$ at $k=20$ |
| Precision@k | $\frac{|\text{Relevant} \cap \text{Retrieved@k}|}{k}$ | $\geq 0.70$ at $k=10$ |
| MRR | $\frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$ | $\geq 0.80$ |
| Provenance Coverage | $\frac{|\text{Chunks with valid provenance}|}{|\text{Total chunks returned}|}$ | $= 1.00$ |
| Latency P99 | End-to-end retrieval time | $\leq 500\text{ms}$ |
| Context Precision | $\frac{\text{Tokens in relevant chunks}}{\text{Total retrieved tokens}}$ | $\geq 0.60$ |

---

## 6. Memory: Stratified Durable State with Hard Memory Walls

### 6.1 The Memory Problem

LLMs are **stateless** per inference call (Theorem 1.1, Property 3). Without an external memory system, every conversation begins from zero, corrections are forgotten, user preferences are lost, and the system cannot learn from its own mistakes.

**However**, indiscriminate memory accumulation introduces its own failure modes:
- **Context pollution:** Stale or incorrect memories degrade generation quality
- **Memory hallucination:** The model "remembers" things that were never validated
- **Privacy violations:** Retaining information beyond its authorized lifetime
- **Contradiction accumulation:** Memories from different contexts conflicting without resolution

### 6.2 Five-Tier Memory Architecture

We formalize memory as a **stratified system** with strict promotion/demotion policies and a **hard memory wall** between ephemeral and durable tiers.

**Definition 6.1 (Memory Tier Hierarchy):**

```
┌─────────────────────────────────────────────────┐
│            TIER 5: PROCEDURAL MEMORY            │
│  "How to do things" — Validated workflows,      │
│   tool usage patterns, successful strategies    │
│  Persistence: Indefinite | Scope: Organizational│
├─────────────────────────────────────────────────┤
│            TIER 4: SEMANTIC MEMORY              │
│  "What is true" — Facts, entity relationships,  │
│   domain knowledge, organizational policies     │
│  Persistence: Indefinite | Scope: Organizational│
├─────────────────────────────────────────────────┤
│            TIER 3: EPISODIC MEMORY              │
│  "What happened" — Validated interaction         │
│   outcomes, corrections, learned preferences    │
│  Persistence: TTL-governed | Scope: User/Project│
╠═════════════════════════════════════════════════╣ ← HARD MEMORY WALL
│            TIER 2: SESSION MEMORY               │
│  Active conversation state, accumulated context │
│  Persistence: Session lifetime | Scope: Session │
├─────────────────────────────────────────────────┤
│            TIER 1: WORKING MEMORY               │
│  Current inference context, tool results,       │
│   intermediate reasoning state                  │
│  Persistence: Single inference | Scope: Request │
└─────────────────────────────────────────────────┘
```

**Definition 6.2 (Hard Memory Wall).** The boundary between Tiers 2 and 3 is a **write-controlled barrier**. Information can only be promoted above the wall through an explicit validation pipeline:

$$\text{Promote}(m) : \text{Tier}_{1,2} \rightarrow \text{Tier}_{3,4,5} \quad \text{iff} \quad \text{Validate}(m) \wedge \text{Deduplicate}(m) \wedge \text{Provenance}(m) \wedge \text{Policy}(m)$$

No information crosses this wall automatically. This is the primary defense against memory hallucination and context pollution.

### 6.3 Memory Item Schema

```protobuf
message MemoryItem {
    string id = 1;                          // UUID
    MemoryTier tier = 2;                    // WORKING | SESSION | EPISODIC | SEMANTIC | PROCEDURAL
    string content = 3;                     // The memory content
    string content_embedding = 4;           // For semantic retrieval
    
    Provenance provenance = 5;              // How this memory was created
    message Provenance {
        string source_interaction_id = 1;   // Originating conversation/task
        string created_by = 2;              // USER | AGENT | SYSTEM | HUMAN_REVIEWER
        Timestamp created_at = 3;
        string validation_method = 4;       // HUMAN_CONFIRMED | TOOL_VERIFIED | INFERRED
        float confidence = 5;               // [0.0, 1.0]
    }
    
    ExpiryPolicy expiry = 6;
    message ExpiryPolicy {
        Timestamp hard_expiry = 1;          // Absolute deletion time
        Duration soft_ttl = 2;              // Time since last access before demotion
        int32 max_access_count = 3;         // Usage-based expiry
    }
    
    repeated string supersedes = 7;         // IDs of memories this replaces
    repeated string contradicts = 8;        // IDs of memories this conflicts with
    AccessControl access = 9;              // Who can read/write this memory
    int32 token_cost = 10;                 // Pre-computed serialization cost
}
```

### 6.4 Memory Write Pipeline (Promotion Algorithm)

**Algorithm 6.1: Memory Promotion with Validation**

```
ALGORITHM PromoteMemory(candidate, target_tier, memory_store)
    // Gate 1: Content Validation
    IF target_tier ≥ EPISODIC:
        validation ← VALIDATE_CONTENT(candidate)
        IF validation.source == TOOL_OUTPUT:
            candidate.provenance.validation_method ← TOOL_VERIFIED
            candidate.provenance.confidence ← 0.9
        ELIF validation.source == HUMAN_CONFIRMATION:
            candidate.provenance.validation_method ← HUMAN_CONFIRMED
            candidate.provenance.confidence ← 0.95
        ELIF validation.source == AGENT_INFERENCE:
            // Agent-inferred memories require higher scrutiny
            IF validation.confidence < 0.8:
                LOG("Memory promotion rejected: low confidence", candidate)
                RETURN REJECTED
            candidate.provenance.validation_method ← INFERRED
        ELSE:
            RETURN REJECTED  // Unvalidated content cannot cross the wall
    
    // Gate 2: Deduplication
    existing ← memory_store.SEMANTIC_SEARCH(
        candidate.content_embedding,
        tier = target_tier,
        similarity_threshold = 0.92
    )
    IF |existing| > 0:
        best_match ← existing[0]
        IF ENTAILS(candidate.content, best_match.content):
            LOG("Duplicate detected, skipping", candidate.id, best_match.id)
            RETURN DEDUPLICATED
        ELIF CONTRADICTS(candidate.content, best_match.content):
            // Contradiction resolution
            IF candidate.provenance.confidence > best_match.provenance.confidence:
                candidate.supersedes ← [best_match.id]
                best_match.status ← SUPERSEDED
            ELSE:
                candidate.contradicts ← [best_match.id]
                FLAG_FOR_HUMAN_REVIEW(candidate, best_match)
                RETURN PENDING_REVIEW
    
    // Gate 3: Policy Compliance
    IF NOT POLICY_CHECK(candidate, target_tier):
        // Check: PII restrictions, retention limits, scope boundaries
        RETURN POLICY_VIOLATION
    
    // Gate 4: Expiry Assignment
    candidate.expiry ← ASSIGN_EXPIRY(target_tier, candidate.provenance)
    // Episodic: TTL = 90 days default
    // Semantic: TTL = indefinite, review_cycle = 180 days
    // Procedural: TTL = indefinite, deprecation_on_failure
    
    // Gate 5: Token Budget Check
    IF memory_store.TOTAL_TOKENS(target_tier) + candidate.token_cost > TIER_BUDGET(target_tier):
        evicted ← memory_store.EVICT_LRU(target_tier, candidate.token_cost)
        LOG("Memory eviction", evicted)
    
    // Commit
    candidate.tier ← target_tier
    candidate.id ← GENERATE_UUID()
    memory_store.WRITE(candidate)
    EMIT_EVENT("memory.promoted", candidate.id, target_tier)
    
    RETURN COMMITTED
```

### 6.5 Memory Read Pipeline (Context Injection)

When the Prefill Compiler requests memory summaries, it executes a **prioritized read** across tiers:

**Algorithm 6.2: Memory Retrieval for Context Injection**

```
ALGORITHM RetrieveMemoryForContext(query, user, session, budget_tokens)
    memories ← []
    remaining_budget ← budget_tokens
    
    // Priority 1: Procedural memories (how-to knowledge)
    IF query.intent ∈ {PROCEDURAL, TOOL_USE}:
        proc_mems ← memory_store.SEARCH(
            tier = PROCEDURAL,
            query_embedding = EMBED(query),
            scope = [ORGANIZATIONAL],
            top_k = 3
        )
        memories.EXTEND(proc_mems)
        remaining_budget -= SUM(m.token_cost for m in proc_mems)
    
    // Priority 2: Relevant episodic memories (past corrections, preferences)
    episodic_mems ← memory_store.SEARCH(
        tier = EPISODIC,
        query_embedding = EMBED(query),
        scope = [user.id, session.project_id],
        top_k = 5,
        recency_weight = 0.3  // Prefer recent episodes
    )
    // Score: relevance * recency * confidence
    episodic_scored ← RANK(episodic_mems,
        key = λm: m.score * DECAY(m.provenance.created_at) * m.provenance.confidence)
    
    FOR m IN episodic_scored:
        IF remaining_budget >= m.token_cost:
            memories.ADD(m)
            remaining_budget -= m.token_cost
    
    // Priority 3: Semantic memories (facts, entity knowledge)
    semantic_mems ← memory_store.SEARCH(
        tier = SEMANTIC,
        query_embedding = EMBED(query),
        scope = [ORGANIZATIONAL, user.department],
        top_k = 5
    )
    FOR m IN semantic_mems:
        IF remaining_budget >= m.token_cost:
            memories.ADD(m)
            remaining_budget -= m.token_cost
    
    // Serialize with provenance
    summary ← SERIALIZE_MEMORY_BLOCK(memories,
                                      format = "STRUCTURED",
                                      include_provenance = TRUE)
    
    RETURN MemorySummary(
        content = summary,
        token_count = budget_tokens - remaining_budget,
        item_count = |memories|,
        tiers_accessed = UNIQUE([m.tier for m in memories])
    )
```

### 6.6 Memory Decay and Maintenance

**Definition 6.3 (Temporal Decay Function).** Episodic memories decay according to a **modified Ebbinghaus curve** gated by access frequency:

$$\text{Strength}(m, t) = \text{confidence}(m) \cdot e^{-\lambda \cdot (t - t_{\text{last\_access}})} \cdot \left(1 + \log(1 + \text{access\_count}(m))\right)$$

where $\lambda$ is the decay rate (configurable per tier).

**Algorithm 6.3: Memory Maintenance Agent (Recurring)**

```
ALGORITHM MemoryMaintenanceAgent()
    // Runs periodically (e.g., daily)
    
    // Phase 1: Expire stale memories
    FOR EACH m IN memory_store.ALL():
        IF m.expiry.hard_expiry < NOW():
            memory_store.DELETE(m.id)
            LOG("Hard expiry", m.id)
        ELIF m.expiry.soft_ttl AND
             (NOW() - m.last_accessed) > m.expiry.soft_ttl:
            IF m.tier == EPISODIC:
                memory_store.DEMOTE(m.id, to=ARCHIVE)
            LOG("Soft TTL demotion", m.id)
    
    // Phase 2: Contradiction detection
    contradictions ← DETECT_CONTRADICTIONS(memory_store,
                                            method = "pairwise_entailment",
                                            sample_rate = 0.1)
    FOR EACH (m1, m2) IN contradictions:
        IF m1.provenance.confidence != m2.provenance.confidence:
            SUPERSEDE(lower_confidence, higher_confidence)
        ELSE:
            FLAG_FOR_HUMAN_REVIEW(m1, m2)
    
    // Phase 3: Compression (merge similar memories)
    clusters ← CLUSTER_MEMORIES(memory_store,
                                 tier = EPISODIC,
                                 similarity_threshold = 0.88)
    FOR EACH cluster IN clusters WHERE |cluster| > 3:
        merged ← LLM.SUMMARIZE(cluster,
                                instruction = "Merge into single factual statement")
        merged.provenance.confidence ← MIN(m.confidence for m in cluster)
        PromoteMemory(merged, SEMANTIC, memory_store)
        FOR m IN cluster:
            memory_store.DELETE(m.id)
    
    EMIT_METRICS("memory.maintenance", {
        expired: count_expired,
        contradictions_found: |contradictions|,
        clusters_merged: |clusters|
    })
```

---

## 7. Tools: First-Class Typed Infrastructure with Governed Mutation

### 7.1 Tools as Typed Protocol Endpoints

In a Context Engineering system, tools are **not** string templates inserted into prompts. They are **first-class typed infrastructure** exposed through discoverable protocol contracts.

**Definition 7.1 (Tool).** A tool $\mathcal{T}$ is a tuple:

$$\mathcal{T} = (\text{name}, \text{description}, \text{InputSchema}, \text{OutputSchema}, \text{SideEffects}, \text{AuthZ}, \text{Timeout}, \text{RetryPolicy})$$

### 7.2 Protocol Layer Design

| Boundary | Protocol | Justification |
|---|---|---|
| User ↔ Application | **JSON-RPC 2.0** | Human-readable, schema-discoverable, stateless request/response |
| Agent ↔ Tool Server | **MCP (Model Context Protocol)** | Standardized discovery, resource/tool/prompt surfaces, SSE transport |
| Internal Service ↔ Service | **gRPC / Protobuf** | Low latency, streaming, typed contracts, deadline propagation |

### 7.3 MCP Tool Server Specification

```protobuf
// MCP Tool Definition (Protobuf representation of MCP schema)
message ToolDefinition {
    string name = 1;                    // e.g., "query_database"
    string description = 2;            // Human and LLM-readable
    JsonSchema input_schema = 3;       // Strict input validation
    JsonSchema output_schema = 4;      // Optional structured output
    
    ToolCapabilities capabilities = 5;
    message ToolCapabilities {
        bool is_read_only = 1;         // No side effects
        bool requires_approval = 2;    // Human-in-the-loop gate
        bool supports_pagination = 3;
        bool supports_streaming = 4;
        TimeoutClass timeout_class = 5; // FAST(<1s) | MEDIUM(<10s) | SLOW(<60s)
        RetryPolicy retry_policy = 6;
    }
    
    AuthorizationRequirement auth = 6;
    message AuthorizationRequirement {
        repeated string required_scopes = 1;  // e.g., ["db:read", "db:write"]
        bool caller_scoped = 2;               // Use caller's credentials, not agent's
    }
    
    AuditConfig audit = 7;
    message AuditConfig {
        bool log_inputs = 1;
        bool log_outputs = 2;
        bool log_caller_identity = 3;
        string audit_sink = 4;        // Where to send audit events
    }
}
```

### 7.4 Tool Invocation with Governance

**Algorithm 7.1: Governed Tool Invocation**

```
ALGORITHM InvokeTool(tool_name, arguments, caller_context, trace_id)
    // Phase 1: Discovery and Validation
    tool ← TOOL_REGISTRY.RESOLVE(tool_name)
    IF tool == NULL: RETURN Error(TOOL_NOT_FOUND, tool_name)
    
    validation ← VALIDATE_INPUT(arguments, tool.input_schema)
    IF NOT validation.valid:
        RETURN Error(INVALID_INPUT, validation.errors)
    
    // Phase 2: Authorization
    IF tool.auth.caller_scoped:
        credentials ← caller_context.credentials  // NOT agent credentials
    ELSE:
        credentials ← SERVICE_ACCOUNT(tool_name)
    
    auth_result ← AUTHORIZE(credentials, tool.auth.required_scopes)
    IF NOT auth_result.permitted:
        RETURN Error(UNAUTHORIZED, auth_result.reason)
    
    // Phase 3: Approval Gate (for state-changing operations)
    IF tool.capabilities.requires_approval AND NOT tool.capabilities.is_read_only:
        approval ← REQUEST_HUMAN_APPROVAL(
            tool_name = tool_name,
            arguments = arguments,
            caller = caller_context.user_id,
            timeout = 300s  // 5 minute approval window
        )
        IF NOT approval.granted:
            RETURN Error(APPROVAL_DENIED, approval.reason)
    
    // Phase 4: Execution with Timeout and Retry
    timeout ← tool.capabilities.timeout_class.duration
    retry_budget ← tool.capabilities.retry_policy
    
    result ← EXECUTE_WITH_RETRY(
        fn = () → tool.server.CALL(arguments, credentials),
        timeout = timeout,
        max_retries = retry_budget.max_attempts,
        backoff = EXPONENTIAL_WITH_JITTER(
            base = retry_budget.initial_backoff,
            max = retry_budget.max_backoff
        ),
        retry_on = retry_budget.retryable_errors,
        circuit_breaker = CIRCUIT_BREAKER(tool_name)
    )
    
    // Phase 5: Output Validation and Audit
    IF tool.output_schema:
        output_valid ← VALIDATE_OUTPUT(result.data, tool.output_schema)
        IF NOT output_valid:
            LOG_WARNING("Tool output schema violation", tool_name, trace_id)
    
    AUDIT_LOG(AuditEntry(
        trace_id = trace_id,
        tool_name = tool_name,
        input = REDACT_PII(arguments) IF tool.audit.log_inputs,
        output = REDACT_PII(result.data) IF tool.audit.log_outputs,
        caller = caller_context.user_id,
        latency = result.duration,
        status = result.status,
        timestamp = NOW()
    ))
    
    RETURN ToolResult(
        data = result.data,
        provenance = {
            source_type: TOOL,
            tool_name: tool_name,
            invocation_time: NOW(),
            trace_id: trace_id
        },
        token_cost = COUNT_TOKENS(SERIALIZE(result.data))
    )
```

### 7.5 Lazy Tool Loading for Token Efficiency

Including all available tool schemas in every context consumes significant token budget. SOTA systems use **lazy tool loading** — only include schemas for tools the agent is likely to need.

**Algorithm 7.2: Contextual Tool Selection**

```
ALGORITHM SelectToolsForContext(query, intent, tool_registry, max_tools=5)
    // Phase 1: Intent-based pre-filtering
    candidate_tools ← tool_registry.FILTER(
        categories = intent.tool_categories,  // e.g., [DATABASE, SEARCH, CALENDAR]
        capabilities = intent.required_capabilities
    )
    
    // Phase 2: Semantic relevance ranking
    query_embedding ← EMBED(query)
    tool_scores ← []
    FOR EACH tool IN candidate_tools:
        tool_embedding ← EMBED(tool.description + " " + tool.name)
        score ← COSINE_SIM(query_embedding, tool_embedding)
        tool_scores.ADD((tool, score))
    
    // Phase 3: Token-budget-aware selection
    selected ← []
    budget_used ← 0
    FOR (tool, score) IN SORT(tool_scores, by=score, desc=TRUE):
        schema_cost ← COUNT_TOKENS(tool.input_schema.serialize())
        IF budget_used + schema_cost ≤ B_tools AND |selected| < max_tools:
            selected.ADD(tool)
            budget_used += schema_cost
    
    RETURN selected
```

**Token savings:** In a registry of 50+ tools, lazy loading typically reduces tool-schema token consumption from 3,000–5,000 tokens to 500–1,000 tokens — a 3–5× improvement in $B_{\text{tools}}$ efficiency.

---

## 8. Agents: Bounded Control Systems with Verification Loops

### 8.1 Agent as Control System

An agent is **not** a prompt with a loop. It is a **bounded control system** that orchestrates the Context Engineering stack to solve tasks through iterative, verifiable execution.

**Definition 8.1 (Agent).** An agent $\mathcal{A}$ is a tuple:

$$\mathcal{A} = (\mathcal{M}_\theta, \Pi, \mathcal{L}_{\text{loop}}, \mathcal{B}_{\text{bounds}}, \mathcal{V}_{\text{verify}}, \mathcal{W}_{\text{workspace}})$$

where:
- $\mathcal{M}_\theta$: The underlying LLM
- $\Pi$: The Context Engineering pipeline (Sections 3–7)
- $\mathcal{L}_{\text{loop}}$: The control loop specification
- $\mathcal{B}_{\text{bounds}}$: Recursion bounds, timeout limits, token budgets
- $\mathcal{V}_{\text{verify}}$: Verification predicates and quality gates
- $\mathcal{W}_{\text{workspace}}$: Isolated execution workspace

### 8.2 The Canonical Agent Loop

**Algorithm 8.1: Bounded Agent Loop with Verification**

```
ALGORITHM AgentLoop(task, agent_config, workspace)
    // Initialization
    state ← AgentState(
        task = task,
        plan = NULL,
        iteration = 0,
        trace = [],
        status = RUNNING
    )
    
    TRY:
        // ═══════════════════════════════════════
        // PHASE 1: PLAN
        // ═══════════════════════════════════════
        context ← PrefillCompiler(
            policy = agent_config.role_policy,
            task = task,
            tools = SelectToolsForContext(task.query, task.intent, TOOL_REGISTRY),
            evidence = [],  // No evidence yet
            memory = RetrieveMemoryForContext(task.query, task.user, task.session,
                                              budget_tokens = 500),
            history = task.session.compressed_history,
            W = agent_config.token_budget
        )
        
        plan ← LLM.GENERATE(context + "Create a step-by-step plan.",
                              output_schema = PlanSchema)
        state.plan ← plan
        state.trace.ADD(TraceEntry(phase="PLAN", output=plan))
        
        // Validate plan feasibility
        IF |plan.steps| > agent_config.max_steps:
            plan ← TRUNCATE_PLAN(plan, agent_config.max_steps)
            LOG_WARNING("Plan truncated", task.id)
        
        // ═══════════════════════════════════════
        // PHASE 2: EXECUTE (Iterative)
        // ═══════════════════════════════════════
        FOR step IN plan.steps:
            state.iteration += 1
            
            // Bound check
            IF state.iteration > agent_config.max_iterations:
                state.status ← BOUNDED_EXIT
                BREAK
            IF ELAPSED(state.start_time) > agent_config.timeout:
                state.status ← TIMEOUT
                BREAK
            
            // ── DECOMPOSE ──
            sub_tasks ← IntentDecompose(step.description,
                                         task.session,
                                         workspace.memory)
            
            // ── RETRIEVE ──
            routed_queries ← RouteQueries(sub_tasks, SOURCE_REGISTRY)
            evidence ← []
            FOR rq IN routed_queries:
                result ← HybridRetrieve(rq)
                evidence.EXTEND(result)
            evidence_ranked ← WeightedRRF(evidence)
            
            // ── ACT ──
            action_context ← PrefillCompiler(
                policy = agent_config.role_policy,
                task = step,
                tools = SelectToolsForContext(step.description, step.intent,
                                              TOOL_REGISTRY),
                evidence = evidence_ranked[:10],  // Top 10 evidence chunks
                memory = RetrieveMemoryForContext(step.description, task.user,
                                                  task.session, budget_tokens=300),
                history = workspace.action_history,
                W = agent_config.token_budget
            )
            
            action ← LLM.GENERATE(action_context,
                                    output_schema = ActionSchema)
            
            IF action.type == TOOL_CALL:
                tool_result ← InvokeTool(action.tool_name,
                                          action.arguments,
                                          task.caller_context,
                                          state.trace_id)
                workspace.action_history.ADD(tool_result)
            ELIF action.type == GENERATE:
                workspace.artifacts.ADD(action.output)
            
            state.trace.ADD(TraceEntry(phase="ACT", step=step, action=action))
            
            // ── VERIFY ──
            verification ← VERIFY_STEP(
                step = step,
                action = action,
                result = tool_result OR action.output,
                evidence = evidence_ranked,
                verification_predicates = agent_config.verify_predicates
            )
            
            state.trace.ADD(TraceEntry(phase="VERIFY", result=verification))
            
            IF verification.status == PASS:
                CONTINUE  // Proceed to next step
            
            // ── CRITIQUE ──
            IF verification.status == FAIL:
                critique ← LLM.GENERATE(
                    context = action_context +
                              FORMAT("Verification failed: {}", verification.reason),
                    instruction = "Identify the root cause and propose a repair."
                )
                state.trace.ADD(TraceEntry(phase="CRITIQUE", output=critique))
                
                // ── REPAIR ──
                repair_attempts ← 0
                WHILE verification.status != PASS AND
                      repair_attempts < agent_config.max_repair_attempts:
                    
                    repaired_action ← LLM.GENERATE(
                        context = action_context + critique +
                                  FORMAT("Repair attempt {}", repair_attempts + 1),
                        output_schema = ActionSchema
                    )
                    
                    IF repaired_action.type == TOOL_CALL:
                        tool_result ← InvokeTool(repaired_action.tool_name,
                                                  repaired_action.arguments,
                                                  task.caller_context,
                                                  state.trace_id)
                    
                    verification ← VERIFY_STEP(step, repaired_action,
                                                tool_result, evidence_ranked,
                                                agent_config.verify_predicates)
                    repair_attempts += 1
                
                IF verification.status != PASS:
                    // Persist failure state for post-mortem
                    state.status ← STEP_FAILED
                    PERSIST_FAILURE_STATE(state, workspace)
                    
                    IF agent_config.on_failure == ABORT:
                        BREAK
                    ELIF agent_config.on_failure == SKIP_AND_CONTINUE:
                        CONTINUE
                    ELIF agent_config.on_failure == COMPENSATE:
                        EXECUTE_COMPENSATING_ACTION(step, workspace)
                        CONTINUE
        
        // ═══════════════════════════════════════
        // PHASE 3: COMMIT
        // ═══════════════════════════════════════
        IF state.status == RUNNING:
            state.status ← COMPLETED
        
        // Synthesize final output
        final_output ← LLM.SYNTHESIZE(
            workspace.artifacts,
            task.original_query,
            output_format = task.required_format
        )
        
        // Memory promotion: extract learnings
        learnings ← EXTRACT_LEARNINGS(state.trace)
        FOR learning IN learnings:
            PromoteMemory(learning, EPISODIC, MEMORY_STORE)
        
        RETURN AgentResult(
            output = final_output,
            status = state.status,
            trace = state.trace,
            metrics = COMPUTE_METRICS(state)
        )
    
    CATCH Exception AS e:
        state.status ← ERROR
        PERSIST_FAILURE_STATE(state, workspace)
        RETURN AgentResult(
            output = NULL,
            status = ERROR,
            error = e,
            trace = state.trace
        )
```

### 8.3 Multi-Agent Orchestration

For complex tasks, multiple specialized agents operate concurrently with **isolation guarantees**.

**Definition 8.2 (Multi-Agent Orchestration).** A multi-agent system $\mathcal{S}$ is:

$$\mathcal{S} = (\{\mathcal{A}_1, \ldots, \mathcal{A}_n\}, \mathcal{O}, \mathcal{L}_{\text{lock}}, \mathcal{M}_{\text{merge}})$$

where:
- $\{\mathcal{A}_i\}$: Specialized agents (Implementer, Verifier, Documenter, Retriever, Performance Analyst)
- $\mathcal{O}$: Orchestrator that assigns work units
- $\mathcal{L}_{\text{lock}}$: Lock/lease manager for shared resources
- $\mathcal{M}_{\text{merge}}$: Merge strategy for combining agent outputs

**Algorithm 8.2: Multi-Agent Orchestration with Lock Discipline**

```
ALGORITHM MultiAgentOrchestrate(task, agent_pool)
    // Decompose into independently claimable work units
    work_units ← DECOMPOSE_TO_WORK_UNITS(task)
    
    // Dependency analysis
    dep_graph ← BUILD_DEPENDENCY_GRAPH(work_units)
    execution_waves ← TOPOLOGICAL_SORT_INTO_WAVES(dep_graph)
    
    results ← {}
    
    FOR wave IN execution_waves:
        // All work units in a wave can execute in parallel
        futures ← []
        
        FOR wu IN wave:
            // Assign to specialized agent
            agent ← SELECT_AGENT(wu.type, agent_pool)
            // {IMPLEMENTATION → ImplementerAgent,
            //  VERIFICATION → VerifierAgent,
            //  DOCUMENTATION → DocumenterAgent, ...}
            
            // Acquire locks on shared resources
            locks ← []
            FOR resource IN wu.required_resources:
                lock ← LOCK_MANAGER.ACQUIRE(
                    resource_id = resource,
                    holder = agent.id,
                    lease_duration = wu.estimated_duration * 2,
                    timeout = 30s
                )
                IF lock == FAILED:
                    RESCHEDULE(wu, next_wave)
                    CONTINUE outer
                locks.ADD(lock)
            
            // Create isolated workspace
            workspace ← CREATE_ISOLATED_WORKSPACE(
                base = task.workspace,
                branch = wu.id,  // Git-like branching for isolation
                inherited_context = results.get(wu.dependencies, {})
            )
            
            // Launch asynchronously
            future ← ASYNC_EXECUTE(
                AgentLoop(wu, agent.config, workspace),
                deadline = wu.deadline
            )
            futures.ADD((wu, future, locks))
        
        // Await wave completion
        FOR (wu, future, locks) IN futures:
            TRY:
                result ← AWAIT(future, timeout = wu.deadline)
                results[wu.id] ← result
                
                // Merge workspace back
                merge_result ← MERGE_WORKSPACE(
                    source = workspace,
                    target = task.workspace,
                    strategy = MERGE_STRATEGY(wu.type),
                    conflict_resolution = AGENT_ASSISTED
                )
                
                IF merge_result.has_conflicts:
                    // Route to human or senior agent
                    RESOLVE_CONFLICTS(merge_result.conflicts)
            
            FINALLY:
                // Always release locks
                FOR lock IN locks:
                    LOCK_MANAGER.RELEASE(lock)
    
    // Final synthesis
    final_output ← SYNTHESIZE_AGENT(results, task)
    RETURN final_output
```

### 8.4 Agent Specialization Roles

| Role | Responsibility | Verification Method |
|---|---|---|
| **Implementer** | Generates code, documents, or data transformations | Unit tests, type checks, schema validation |
| **Verifier** | Reviews implementer output against requirements | Predicate checking, test execution, consistency analysis |
| **Retriever** | Manages complex multi-hop retrieval tasks | Recall/precision metrics, provenance completeness |
| **Documenter** | Produces documentation, summaries, change logs | Coverage analysis, readability scores |
| **Performance Analyst** | Profiles latency, cost, token efficiency | SLA compliance, budget adherence |
| **Critic** | Adversarial review of other agents' outputs | Finds counterexamples, edge cases, contradictions |

### 8.5 Concurrency Control Invariants

**Theorem 8.1 (Safe Concurrency).** Parallel agent execution is safe iff:

$$\forall\ \mathcal{A}_i, \mathcal{A}_j\ \text{executing in parallel}: \quad \mathcal{W}(\mathcal{A}_i) \cap \mathcal{W}(\mathcal{A}_j) = \emptyset \quad \vee \quad \text{Lock}(\mathcal{W}(\mathcal{A}_i) \cap \mathcal{W}(\mathcal{A}_j)) = \text{held}$$

where $\mathcal{W}(\mathcal{A})$ is the set of resources agent $\mathcal{A}$ writes to.

That is, either write sets are disjoint (no contention), or shared write targets are protected by exclusive locks.

**Merge Entropy Bound:** The merge complexity after parallel execution is:

$$H_{\text{merge}} = \sum_{r \in \text{SharedResources}} \log_2(1 + |\text{ConflictingWrites}(r)|)$$

The orchestrator must ensure $H_{\text{merge}} \leq H_{\text{max}}$ before authorizing parallel execution; otherwise, serialize.

---

## 9. Integrated System Architecture and Data Flow

### 9.1 End-to-End Request Processing

The following formalizes the complete data flow from user query to response, mapping to the 8-step diagram from the original system:

```
USER QUERY (JSON-RPC)
    │
    ▼
[1] AGENT RECEIVES QUERY
    │
    ├──[2]──► QUERY AUGMENTATION
    │           ├── Coreference resolution (against session history)
    │           ├── Intent classification
    │           ├── Multi-hop decomposition
    │           ├── HyDE + multi-representation expansion
    │           └── Source routing with latency tiers
    │
    ├──[3]──► RETRIEVAL ENGINE (RAG + Vector Search)
    │           ├── Sparse retrieval (SPLADE)
    │           ├── Dense retrieval (ColBERT v2 / E5)
    │           ├── Graph augmentation (multi-hop traversal)
    │           ├── Metadata filtering (pre-retrieval)
    │           ├── Weighted RRF fusion
    │           └── Provenance tagging on every chunk
    │
    ├──[7→]──► MEMORY READ
    │           ├── Procedural (how-to) memories
    │           ├── Episodic (past corrections, preferences)
    │           ├── Semantic (facts, entity knowledge)
    │           └── Budget-aware selection & serialization
    │
    ├──────► PREFILL COMPILER
    │           ├── Token budget allocation
    │           ├── Priority-ranked assembly
    │           ├── Deterministic section composition
    │           └── Invariant verification
    │
    ├──[5]──► TOOL INVOCATION (MCP / gRPC)
    │           ├── Lazy tool selection
    │           ├── Input schema validation
    │           ├── Authorization (caller-scoped)
    │           ├── Approval gate (if mutating)
    │           ├── Execution with retry + circuit breaker
    │   [6]◄───├── Output validation
    │           └── Audit logging
    │
    ├──────► AGENT LOOP VERIFICATION
    │           ├── Verify action against evidence
    │           ├── Critique if failed
    │           ├── Repair with bounded retries
    │           └── Commit or compensate
    │
    ├──[←7]──► MEMORY WRITE
    │           ├── Extract learnings from trace
    │           ├── Validation gate
    │           ├── Deduplication
    │           ├── Contradiction resolution
    │           ├── Policy compliance
    │           └── Promote to episodic/semantic/procedural
    │
    ▼
[8] RESPONSE TO USER (with provenance citations)
```

### 9.2 System Invariants (Enforced Mechanically)

| Invariant | Enforcement Mechanism |
|---|---|
| Token budget never exceeded | Prefill Compiler ASSERT at assembly time |
| Every evidence chunk has provenance | Schema validation on Evidence type |
| No unvalidated memory promotion | PromoteMemory 5-gate pipeline |
| Tool mutations are human-interruptible | Approval gate in InvokeTool |
| Agent recursion is bounded | max_iterations + timeout in AgentLoop |
| Concurrent agents don't corrupt shared state | Lock discipline + workspace isolation |
| Failed states are persisted | PERSIST_FAILURE_STATE in catch blocks |
| All tool invocations are audited | AuditConfig on every ToolDefinition |

---

## 10. Operational Constraints: Reliability, Cost, Latency, and Observability

### 10.1 Reliability Engineering

**10.1.1 Retry Budget with Exponential Backoff and Jitter**

$$\text{Delay}(n) = \min\!\left(\text{MaxBackoff},\ \text{Base} \cdot 2^n + \text{Uniform}(0, \text{Jitter})\right)$$

where $n$ is the retry attempt number.

**Budget:** Total retries across all tool calls in a single agent execution are bounded:

$$\sum_{\text{tools}} \text{retries}(t_i) \leq R_{\text{max}} \quad (R_{\text{max}} \text{ configurable, default } 10)$$

**10.1.2 Circuit Breaker**

```
ALGORITHM CircuitBreaker(tool_name)
    state ← LOAD_STATE(tool_name)  // CLOSED | OPEN | HALF_OPEN
    
    IF state == OPEN:
        IF NOW() > state.open_until:
            state ← HALF_OPEN
        ELSE:
            RETURN Error(CIRCUIT_OPEN,
                         "Tool {} unavailable, retry after {}",
                         tool_name, state.open_until)
    
    IF state == HALF_OPEN:
        result ← TRY_EXECUTE(tool_name)
        IF result.success:
            state ← CLOSED
            RESET_FAILURE_COUNT(tool_name)
        ELSE:
            state ← OPEN
            state.open_until ← NOW() + BACKOFF_DURATION
    
    // CLOSED: normal execution with failure tracking
    result ← EXECUTE(tool_name)
    IF result.failure:
        INCREMENT_FAILURE_COUNT(tool_name)
        IF FAILURE_COUNT(tool_name) > THRESHOLD:
            state ← OPEN
            state.open_until ← NOW() + BACKOFF_DURATION
    
    SAVE_STATE(tool_name, state)
    RETURN result
```

**10.1.3 Idempotency**

Every state-changing operation must carry an **idempotency key**:

$$\text{IdempotencyKey} = \text{SHA256}(\text{tool\_name} \| \text{arguments} \| \text{caller\_id} \| \text{logical\_timestamp})$$

Tool servers check this key before execution; if the operation was already completed, return the cached result without re-execution.

### 10.2 Cost Optimization

**10.2.1 Token Cost Model**

$$\text{Cost}(\text{request}) = \underbrace{|C| \cdot p_{\text{input}}}_{\text{Context cost}} + \underbrace{|O| \cdot p_{\text{output}}}_{\text{Generation cost}} + \underbrace{\sum_{t} \text{ToolCost}(t)}_{\text{Tool invocation cost}}$$

where $p_{\text{input}}$ and $p_{\text{output}}$ are per-token prices.

**Optimization Strategies:**

| Strategy | Mechanism | Typical Savings |
|---|---|---|
| Context compression | Remove low-utility tokens from evidence | 20–40% input cost |
| Lazy tool loading | Include only relevant tool schemas | 3–5× tool schema cost |
| Tiered model routing | Use cheaper models for simple sub-tasks | 50–70% for routine steps |
| Cache retrieval results | Hash-based cache for repeated queries | 30–60% retrieval cost |
| Prompt caching (API-level) | Anthropic/OpenAI prompt caching | 50–90% for repeated prefixes |
| History compression | Summarize older conversation turns | 40–60% history cost |

**10.2.2 Tiered Model Routing**

$$\text{ModelSelect}(\text{task}) = \begin{cases} \mathcal{M}_{\text{small}} & \text{if complexity}(\text{task}) < \theta_1 \\ \mathcal{M}_{\text{medium}} & \text{if } \theta_1 \leq \text{complexity}(\text{task}) < \theta_2 \\ \mathcal{M}_{\text{large}} & \text{if complexity}(\text{task}) \geq \theta_2 \end{cases}$$

where complexity is estimated from task decomposition depth, required tool count, and evidence volume.

### 10.3 Latency Engineering

**10.3.1 Latency Budget Decomposition**

For a target end-to-end latency $L_{\text{target}}$ (e.g., 5 seconds):

$$L_{\text{target}} = L_{\text{QA}} + L_{\text{retrieval}} + L_{\text{compile}} + L_{\text{LLM}} + L_{\text{tools}} + L_{\text{overhead}}$$

| Component | Budget | Technique |
|---|---|---|
| $L_{\text{QA}}$ | 200ms | Single fast LLM call or classifier |
| $L_{\text{retrieval}}$ | 500ms | Parallel retrieval across sources, ANN index |
| $L_{\text{compile}}$ | 50ms | Deterministic assembly, pre-computed templates |
| $L_{\text{LLM}}$ | 3,000ms | Streaming output, model selection |
| $L_{\text{tools}}$ | 1,000ms | Parallel invocation where independent |
| $L_{\text{overhead}}$ | 250ms | Network, serialization, logging |

**10.3.2 Parallel Retrieval Execution**

```
ALGORITHM ParallelRetrieve(routed_queries, global_timeout)
    futures ← {}
    FOR rq IN routed_queries:
        futures[rq.id] ← ASYNC_EXECUTE(
            HybridRetrieve(rq),
            timeout = MIN(rq.timeout, global_timeout)
        )
    
    results ← []
    FOR (id, future) IN futures:
        TRY:
            result ← AWAIT(future, timeout = global_timeout)
            results.EXTEND(result)
        CATCH TimeoutError:
            LOG_WARNING("Retrieval timeout", id)
            // Graceful degradation: proceed without this source
            EMIT_METRIC("retrieval.timeout", source=id)
    
    RETURN results
```

### 10.4 Observability

**10.4.1 Distributed Tracing**

Every request generates a **trace** following OpenTelemetry conventions:

```
Trace: request_abc123
├── Span: query_augmentation (200ms)
│   ├── Span: coreference_resolution (50ms)
│   ├── Span: intent_classification (80ms)
│   └── Span: query_routing (70ms)
├── Span: retrieval (450ms)
│   ├── Span: sparse_retrieval (120ms)
│   ├── Span: dense_retrieval (300ms)  // parallel
│   ├── Span: graph_augmentation (150ms)  // parallel
│   └── Span: rank_fusion (30ms)
├── Span: memory_read (80ms)
├── Span: prefill_compile (40ms)
│   └── Attribute: token_count=3847, budget_remaining=4153
├── Span: llm_generation (2800ms)
│   └── Attribute: model=gpt-4o, input_tokens=3847, output_tokens=512
├── Span: verification (200ms)
└── Span: memory_write (100ms)
    └── Attribute: items_promoted=1, tier=EPISODIC
```

**10.4.2 Key Metrics Dashboard**

| Metric | Type | Alert Threshold |
|---|---|---|
| `request.latency.p99` | Histogram | > 10s |
| `retrieval.recall` | Gauge | < 0.80 |
| `context.token_utilization` | Gauge | > 0.95 (waste) or < 0.50 (underuse) |
| `hallucination.rate` | Counter (sampled) | > 0.05 |
| `tool.error_rate` | Counter | > 0.10 |
| `memory.promotion_rate` | Counter | Anomaly detection |
| `agent.repair_rate` | Counter | > 0.30 (quality regression) |
| `cost.per_request` | Histogram | > budget threshold |
| `circuit_breaker.open_count` | Gauge | > 0 (alert) |

### 10.5 Evaluation Infrastructure

**Definition 10.1 (Continuous Evaluation Pipeline).** Human corrections, failed traces, and production regressions are converted into **reusable evaluation assets**:

```
ALGORITHM BuildEvalFromProduction(failed_traces, human_corrections)
    eval_suite ← LOAD_EXISTING_SUITE("production_regression")
    
    FOR trace IN failed_traces:
        test_case ← EvalCase(
            input = trace.original_query,
            expected_behavior = trace.human_correction OR trace.expected_output,
            failure_mode = trace.failure_classification,
            context_snapshot = trace.context_at_failure,
            tags = [trace.failure_category, trace.agent_role]
        )
        eval_suite.ADD(test_case)
    
    FOR correction IN human_corrections:
        test_case ← EvalCase(
            input = correction.original_query,
            expected_output = correction.corrected_output,
            grading_criteria = correction.reviewer_criteria,
            tags = ["human_corrected", correction.category]
        )
        eval_suite.ADD(test_case)
    
    // Deduplicate and version
    eval_suite.DEDUPLICATE(similarity_threshold = 0.90)
    eval_suite.VERSION_INCREMENT()
    eval_suite.PUBLISH_TO_CI()
    
    RETURN eval_suite
```

**CI/CD Integration:** No prompt version, retrieval configuration change, or memory policy update is deployed without passing the evaluation suite with:

$$\text{Score}_{\text{new}} \geq \text{Score}_{\text{baseline}} - \epsilon_{\text{regression}}$$

where $\epsilon_{\text{regression}}$ is the maximum acceptable quality regression (typically 0.02 or 2%).

---

## 11. Conclusion

### 11.1 Summary of the Context Engineering Stack

Context Engineering is not a technique — it is a **discipline** that transforms a stateless, isolated LLM into a reliable, grounded, production-grade application through systematic architecture.

| Layer | Function | Key Principle |
|---|---|---|
| **Prompting** | Configure model behavior | Compiled artifacts, not artisanal text |
| **Query Augmentation** | Transform user intent for retrieval | Decompose, rewrite, route — never search naively |
| **Retrieval** | Produce provenance-tagged evidence | Hybrid, ranked, budget-aware — never anonymous blobs |
| **Memory** | Maintain durable, validated state | Hard memory wall, 5-tier hierarchy, validated writes only |
| **Tools** | Enable real-world interaction | Typed contracts, governed mutation, least privilege |
| **Agents** | Orchestrate end-to-end execution | Bounded loops, verification gates, failure persistence |
| **Infrastructure** | Ensure reliability at scale | Circuit breakers, observability, continuous evaluation |

### 11.2 The Core Equation

The quality of an LLM-based system is determined by:

$$\text{SystemQuality} = f\!\left(\text{ModelCapability} \times \text{ContextQuality} \times \text{VerificationCoverage} \times \text{OperationalReliability}\right)$$

Context Engineering maximizes the second factor. The agent loop maximizes the third. Production infrastructure maximizes the fourth. Together, they transform $\mathcal{M}_\theta$ from a brilliant but unreliable oracle into a **trustworthy, observable, and continuously improving system**.

### 11.3 Design Principles (Axiomatic)

1. **Engineer context, not prompts.** The prompt is the last mile — the system is the pipeline.
2. **Every token is a cost.** Budget explicitly; waste nothing.
3. **Every claim needs provenance.** If you can't cite it, don't trust it.
4. **Memory requires validation.** Unvalidated persistence is a hallucination amplifier.
5. **Tools require governance.** Uncontrolled mutation is a production incident.
6. **Agents require bounds.** Unbounded loops are denial-of-service against your own system.
7. **Quality requires measurement.** If you can't eval it, you can't ship it.
8. **Reliability requires mechanism.** Hope is not an engineering strategy.

---

*This report constitutes the complete architectural specification for a production-grade Context Engineering platform. Every subsystem is defined with typed contracts, formal invariants, algorithmic specifications, and measurable quality gates sufficient for implementation by a principal-level engineering team.*